# 🧬 Diccionario humano de citocinas: una práctica interactiva de transcriptómica

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/garridolab/cytokines-practical/blob/main/CytokineDictionary_Practical_v3_interactive_es.ipynb)

**Basado en:** Oesinghaus *et al.* (2025), *A single-cell cytokine dictionary of human peripheral blood*, bioRxiv
([doi:10.64898/2025.12.12.693897](https://doi.org/10.64898/2025.12.12.693897)).

En el artículo, ~9,7 millones de células sanguíneas humanas (PBMC) de **12 donantes** se estimularon con **90 citocinas** (o con un control de PBS)
y se midieron por RNA-seq de célula única. Aquí trabajas con una versión **pseudobulk** de los datos: para cada *donante × citocina × tipo celular*
se **sumaron** los conteos de todas las células en un único perfil de expresión tipo bulk. Usamos **cuatro tipos celulares** (células T CD4, células T CD8, células NK, monocitos CD14),
**87 citocinas** y el **control PBS**.

### 🎛 No vas a escribir código
Esta práctica es **interactiva**. Cada ejercicio te da unos pocos **controles** (menús, deslizadores, cajas) y una **figura que se redibuja sola** cada vez que cambias uno.
Tu trabajo es **jugar, observar e interpretar**: ¿qué cambia? ¿por qué?

| Parte | Ejercicios | Tiempo |
|:--:|---|:--:|
| **A · Calentamiento en Python** | **1** Variables · **2** Gráficos básicos | 12 min |
| **B · Conoce los datos** | **3** El objeto AnnData · **4** Distribuciones con diagramas de cajas | 18 min |
| **C · Normalización** | **5** Conteos crudos vs CPM · **6** El logaritmo | 18 min |
| **D · Patrones en los datos** | **7** Mapas de calor · **8** PCA: la idea · **9** PCA: todas las muestras · **10** PCA: un tipo celular | 38 min |
| **E · Expresión diferencial** | **11** Volcano plots · **12** Comparar dos respuestas | 22 min |
| | Cierre y cuestionario | 7 min |

### Cómo usar este cuaderno

| | |
|:--:|---|
| ▶ | **Ejecuta una celda** con el botón ▶ de su izquierda (o haz clic y pulsa **Shift + Enter**). Ejecuta primero la **celda de preparación** y después cada celda de ejercicio *una vez*: aparecen los controles. |
| 🎛 | **Cambia un control y espera un segundo**: figuras y tablas se actualizan solas. No hace falta volver a ejecutar la celda. |
| 🖱 | En una lista, **Ctrl/⌘ + clic** selecciona varios elementos. En las cajas de texto, pulsa **Enter** al terminar de escribir. |
| ❓ | Cada ejercicio termina con **preguntas**. Piensa (o coméntalo con tu compañero) *antes* de abrir la 💡 respuesta. |
| 🔁 | Si algo se ve raro, vuelve a ejecutar la celda del ejercicio. Si el cuaderno se reinició, ejecuta de nuevo la celda de preparación. |

> 💡 En Google Colab el código de las celdas grises está oculto a propósito. Si tienes curiosidad, haz doble clic en el título de la celda para verlo.
> Los números usan punto decimal (0.93) y coma para los millares (1,353), igual que las figuras.


---
# ⚙️ Preparación: ejecuta esta celda primero

Instala lo que falte, descarga los datos (≈ 500 MB, alrededor de un minuto en Colab), los normaliza, calcula los PCA y carga los resultados de expresión diferencial
calculados de antemano con DESeq2. **Solo la ejecutas una vez** (y otra vez si el cuaderno se reinicia). Espera a que aparezca **✅ Listo**.

In [ ]:
#@title ▶ Preparación: ejecuta esta celda primero (unos 2 minutos) { display-mode: "form" }
# ── Instala lo que falte (Google Colab ya trae casi todo) ──────────────────────────
import importlib.util, shutil, subprocess, sys

REQUIRED = {                       # import name -> PyPI package name
    "numpy": "numpy", "pandas": "pandas", "scipy": "scipy", "matplotlib": "matplotlib", "anndata": "anndata",
    "ipywidgets": "ipywidgets", "jinja2": "jinja2", "gdown": "gdown", "adjustText": "adjustText",
}
missing = [pkg for mod, pkg in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    print(f"📦 Instalando: {', '.join(missing)} …")
    if shutil.which("uv"):
        cmd = ["uv", "pip", "install", "--python", sys.executable, *missing]
    else:
        cmd = [sys.executable, "-m", "pip", "install", "-q", *missing]
    subprocess.check_call(cmd)
    importlib.invalidate_caches()


# ══════════════════════════════════════════════════════════════════════════════════════════════════
#  00_core.py: imports, constants, colours and plot style
#  (The files of tools/lib are concatenated, in alphabetical order, into the hidden setup cell of the notebook.)
# ══════════════════════════════════════════════════════════════════════════════════════════════════
import os, re, io, sys, json, time, math, html, hashlib, difflib, warnings, itertools
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap
from scipy import stats
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

warnings.filterwarnings("ignore")


# ── the dataset ────────────────────────────────────────────────────────────────────────────────────
GDRIVE_FILE_ID = "1thFoQnNVVqgKvspg5b5k2fBgjk4bEqAn"          # pseudobulk .h5ad on Google Drive ("anyone with the link")
DATA_FILENAME = "pseudobulks_4ct_proteincoding.h5ad"
DE_CACHE_FILENAME = "de_results.npz"                           # pre-computed DESeq2 results (all cytokines x all cell types)
DE_CACHE_URL = "https://raw.githubusercontent.com/garridolab/cytokines-practical/main/cache/" + DE_CACHE_FILENAME

DISPLAY = {}      # display names of a few words that are also data keys (filled in by the Spanish build)


def disp(word):
    return DISPLAY.get(word, word)


CELL_TYPES = ["CD4_T_cell", "CD8_T_cell", "NK_CD56low", "CD14_Mono"]
CT_LABEL = {"CD4_T_cell": 'Célula T CD4', "CD8_T_cell": 'Célula T CD8', "NK_CD56low": 'NK CD56low', "CD14_Mono": 'Monocito CD14'}
CT_SHORT = {"CD4_T_cell": "CD4 T", "CD8_T_cell": "CD8 T", "NK_CD56low": "NK", "CD14_Mono": "Mono"}
CT_COLORS = {"CD4_T_cell": "#2a78d6", "CD8_T_cell": "#008300", "NK_CD56low": "#eda100", "CD14_Mono": "#e87ba4"}
CT_MARKERS = {"CD4_T_cell": "o", "CD8_T_cell": "s", "NK_CD56low": "^", "CD14_Mono": "D"}
DONORS = [f"Donor{i}" for i in range(1, 13)]

# the expression filter of the practical: at least 10 counts in at least 12 samples (12 = the size of one group)
MIN_COUNT, MIN_SAMPLES = 10, 12
N_HVG = 2000                                                    # highly variable genes used for every PCA
PADJ_DEFAULT, LFC_DEFAULT = 0.05, 0.5                           # default thresholds of the volcano plots

# ── colours: "direction" is blue (down) < grey < red (up) ─────────────────────────────────────────────
UP_COLOR, DOWN_COLOR, NS_COLOR, PBS_COLOR = "#e34948", "#2a78d6", "#c3c2b7", "#8a8984"
BLUE, ORANGE, GREEN, YELLOW, PINK, PURPLE = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#6250d6"
DARK = "#2b2a28"
DIVERGING = LinearSegmentedColormap.from_list("div", [DOWN_COLOR, "#f0efec", UP_COLOR])
SEQUENTIAL = LinearSegmentedColormap.from_list("seq", ["#f4f8fd", "#86b6ef", "#2a78d6", "#0d366b"])
SEQ_POINTS = LinearSegmentedColormap.from_list("seqp", ["#cde2fb", "#2a78d6", "#0d366b"])
DONOR_HUES = [BLUE, ORANGE, GREEN, YELLOW, PINK, PURPLE]       # 12 donors = 6 hues x 2 marker shapes
CYT_HUES = [BLUE, ORANGE, GREEN, YELLOW, PINK, PURPLE, "#008300", "#8a5a00"]
CYT_MARKERS = ["o", "s", "^", "D", "v", "P", "X", "h"]

plt.rcParams.update({
    "figure.dpi": 100, "savefig.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
    "axes.titleweight": "bold", "axes.titlesize": 11, "axes.labelsize": 10, "legend.frameon": False,
    "legend.fontsize": 8.5, "xtick.labelsize": 9, "ytick.labelsize": 9, "figure.autolayout": False,
})


# ══ 05_immune_genes.py: gene list of the DE exercises ═══

# ── the genes of the differential-expression exercises (11, 12): immune / cytokine-signalling genes ────────────────────────
# Union of four gene sets from Enrichr: GO BP "Cytokine-Mediated Signaling Pathway" (GO:0019221), GO BP "T Cell Activation" (GO:0042110),
# MSigDB Hallmark "Interferon Alpha Response" and "Interferon Gamma Response" (542 symbols; ~506 are expressed in our data).
IMMUNE_GENES = """
ACTN4 ADAR ADIPOQ ADIPOR1 ADIPOR2 AIM2 AKT1 APOL6 APPL1 APPL2 ARID5B ARL4A AUTS2 B2M BANK1 BATF2 BIRC2 BIRC3
BPGM BST2 BTG1 C1R C1S CARD14 CASP1 CASP3 CASP4 CASP7 CASP8 CBL CCL1 CCL11 CCL13 CCL14 CCL15 CCL16 CCL18 CCL19
CCL2 CCL21 CCL22 CCL23 CCL24 CCL25 CCL26 CCL3 CCL3L1 CCL4 CCL4L1 CCL5 CCL7 CCL8 CCM2 CCR1 CCR2 CCR7 CCRL2 CD1C
CD2 CD247 CD27 CD274 CD276 CD28 CD300LF CD38 CD3D CD3E CD4 CD40 CD44 CD47 CD69 CD7 CD70 CD74 CD80 CD84 CD86
CD8A CD8B CDKN1A CEACAM1 CEACAM21 CEBPA CFB CFH CHD7 CHUK CIITA CISH CLCF1 CLEC7A CLSTN1 CMKLR1 CMPK2 CMTR1
CNP CNTF CNTFR COMMD7 CRLF1 CRLF2 CRTAM CSF1 CSF1R CSF2 CSF2RA CSF2RB CSF3 CSF3R CSNK2B CTPS1 CTR9 CTSG CUL4A
CX3CL1 CXADR CXCL10 CXCL11 CXCL12 CXCL9 CXCR2 DCAF12 DDOST DDX58 DDX60 DHX58 DLL4 DPP4 DUOX1 DUOX2 EBI3 EDA
EDA2R EDAR EDN2 EGR1 EIF2AK2 EIF2AK4 EIF4E3 EIF5AL1 ELF1 EPOR EPSTI1 EPX EREG F2RL1 F3 FAS FCER1G FCGR1A FER
FGL2 FKBP1A FLT3 FOXC1 FOXO3 FOXP3 FPR1 GATA3 GBP2 GBP4 GBP6 GCH1 GH1 GHR GMPR GPR18 GPR35 GPR75 GPR89A GRB2
GREM2 GZMA HAX1 HCK HCLS1 HELZ2 HERC6 HIF1A HLA-A HLA-B HLA-C HLA-DMA HLA-DQA1 HLA-DRB1 HLA-G HSPD1 ICAM1
ICOSLG IDO1 IFI27 IFI30 IFI35 IFI44 IFI44L IFIH1 IFIT1 IFIT2 IFIT3 IFITM1 IFITM2 IFITM3 IFNA1 IFNA10 IFNA14
IFNA16 IFNA17 IFNA2 IFNA21 IFNA4 IFNA5 IFNA6 IFNA7 IFNA8 IFNAR2 IFNB1 IFNE IFNGR1 IFNGR2 IFNK IFNLR1 IFNW1
IKBKB IL10 IL10RA IL10RB IL11 IL11RA IL12A IL12B IL12RB1 IL12RB2 IL13 IL13RA1 IL13RA2 IL15 IL15RA IL16 IL17F
IL17RA IL17RC IL18 IL18BP IL18R1 IL18RAP IL1A IL1B IL1F10 IL1R1 IL1RAP IL1RL1 IL2 IL20RA IL20RB IL21R IL22RA1
IL22RA2 IL23R IL2RA IL2RB IL2RG IL3 IL31RA IL33 IL34 IL36A IL36B IL36G IL37 IL3RA IL4 IL4R IL5 IL5RA IL6 IL6R
IL6ST IL7 IL7R IL9 IL9R ILK INS IRAK1 IRAK2 IRAK3 IRAK4 IRF1 IRF2 IRF4 IRF5 IRF7 IRF8 IRF9 IRS1 ISG15 ISG20
ISOC1 ITGB7 ITK JAG2 JAGN1 JAK1 JAK2 JAK3 JAML KIF13B KIT KLRK1 LAMP3 LAP3 LATS2 LCK LCP2 LEP LEPROT LFNG
LGALS3BP LIFR LILRA1 LILRA2 LILRA3 LILRA4 LILRA5 LILRA6 LILRB1 LILRB2 LILRB4 LILRB5 LIMS1 LMBR1L LPAR6 LRP8
LY6E LY9 LYN LYSMD2 MAP3K7 MAPK3 MARCHF1 MDK METTL7B MICA MICB MOV10 MSN MT2A MT3 MTHFD2 MVB12A MVP MX1 MX2
MYD88 NAMPT NCK1 NCK2 NCOA3 NCOA7 NCOR2 NFKB1 NFKBIA NHEJ1 NLRC3 NLRC5 NMI NOD1 NUB1 NUMBL NUP93 OAS1 OAS2
OAS3 OASL OGFR OSMR P2RY14 P3R3URF P4HB PARP12 PARP14 PARP9 PDE4B PDGFB PELI1 PF4 PFKP PIK3CG PIK3R1 PIK3R2
PIK3R3 PIM1 PLA2G4A PLCB1 PLP2 PLSCR1 PLVAP PML PNP PNPT1 PPP3CA PPP3CB PRKCA PRL PRLR PROCR PSMA2 PSMA3
PSMB10 PSMB2 PSMB8 PSMB9 PSME1 PSME2 PTGS2 PTK2B PTPN1 PTPN11 PTPN2 PTPN22 PTPN6 PTPRC PTPRJ PYCARD RAB29
RAPGEF6 RASGRP1 RBCK1 RBX1 RELA RHEX RHOH RIPK1 RIPK2 RNF213 RNF31 RPS6KA4 RPS6KA5 RRAGA RSAD2 RTP4 SAMD9
SAMD9L SAMHD1 SCRIB SECTM1 SELL SELP SERPING1 SIRT1 SLA2 SLAMF6 SLAMF7 SLAMF9 SLC25A28 SMAD4 SOCS1 SOCS2 SOCS3
SOCS4 SOCS5 SOD2 SOS1 SOX4 SP110 SPI1 SPNS1 SPPL2A SRC SRI SSPN ST18 ST3GAL5 ST8SIA4 STAT1 STAT2 STAT3 STAT4
STAT5A STAT5B STAT6 SYK TANK TAP1 TAPBP TARP TDRD7 TENT5A TFF2 TMEM140 TMSB4X TNF TNFAIP2 TNFAIP3 TNFAIP6
TNFRSF11A TNFRSF1A TNFRSF1B TNFRSF4 TNFSF10 TNFSF11 TNFSF14 TNFSF18 TNFSF4 TNIP2 TOLLIP TOR1B TOX4 TP53 TRADD
TRAF1 TRAF3 TRAF3IP2 TRAF6 TRAFD1 TREML2 TRIM14 TRIM21 TRIM25 TRIM26 TRIM5 TSLP TXNDC17 TXNIP TYK2 UBA7 UBE2L6
UGCG UPP1 USP18 VAMP5 VAMP8 VCAM1 WARS1 XAF1 XCL1 XCL2 ZAP70 ZBP1 ZNFX1
""".split()


# ══════════════════════════════════════════════════════════════════════════════════════════════════
#  10_data.py: download, tidy-up and pre-computation (everything slow happens here, once)
# ══════════════════════════════════════════════════════════════════════════════════════════════════

# ── files ───────────────────────────────────────────────────────────────────────────────────────────
def _search_dirs():
    here = os.getcwd()
    dirs = [here, os.path.join(here, "cache"), os.path.dirname(here), "/content", os.path.expanduser("~")]
    return [d for d in dict.fromkeys(dirs) if os.path.isdir(d)]


def find_file(filename):
    """Path of `filename` if it already exists next to the notebook (or in /content on Colab), else None."""
    for d in _search_dirs():
        p = os.path.join(d, filename)
        if os.path.exists(p):
            return p
    return None


def download_dir():
    return "/content" if os.path.isdir("/content") else os.getcwd()


def get_h5ad(path=None, verbose=True):
    """Path to the pseudobulk .h5ad; downloads it from Google Drive the first time."""
    path = path or find_file(DATA_FILENAME)
    if path and os.path.exists(path):
        return path
    import gdown
    path = os.path.join(download_dir(), DATA_FILENAME)
    if verbose:
        print('⬇️  Descargando los datos de Google Drive (≈ 500 MB, alrededor de un minuto) …')
    out = gdown.download(f"https://drive.google.com/uc?id={GDRIVE_FILE_ID}", path, quiet=not verbose)
    if out is None or not os.path.exists(path):
        raise RuntimeError('La descarga ha fallado (¿cuota de Google Drive?). Espera un minuto y vuelve a ejecutar esta celda, o pide el archivo al profesor.')
    return path


def get_de_cache(verbose=True):
    """Path to the pre-computed DESeq2 results, or None if they cannot be found/downloaded (the notebook then computes on demand)."""
    p = find_file(DE_CACHE_FILENAME)
    if p:
        return p
    try:
        import urllib.request
        target = os.path.join(download_dir(), DE_CACHE_FILENAME)
        if verbose:
            print('⬇️  Descargando los resultados de expresión diferencial precalculados …')
        urllib.request.urlretrieve(DE_CACHE_URL, target)
        return target
    except Exception as err:                                      # offline, or the file is not published (yet)
        if verbose:
            print(f"ℹ️  Resultados de expresión diferencial precalculados no disponibles ({type(err).__name__}): se calcularán cuando haga falta (≈ 15 s cada uno).")
        return None


# ── tidy-up of the AnnData object ─────────────────────────────────────────────────────────────────
def is_expressed(counts, min_count=MIN_COUNT, min_samples=MIN_SAMPLES):
    """For a samples × genes matrix of raw counts: True for the genes with at least `min_count` counts in at least `min_samples` samples."""
    return (np.asarray(counts) >= min_count).sum(axis=0) >= min_samples


def load_adata(path):
    """Read the .h5ad and return the tidy AnnData of the practical:
    X = raw counts (dense), obs = sample metadata, var = gene metadata, layers: `cpm` and `log2cpm` (normalised versions of X)."""
    raw = ad.read_h5ad(path)
    # 1) the PBS control was measured in 6 wells and the file stores each well *plus* their total: keep the total only
    is_pbs_well = ((raw.obs["cytokine"] == "PBS") & raw.obs["bc1_well"].notna()).values
    raw = raw[~is_pbs_well]
    # 2) sample metadata, with the order of the categories fixed, and a unique readable name:  donor|cell type|cytokine
    obs = raw.obs[["donor", "cytokine", "treatment", "cell_type", "psbulk_n_cells"]].copy()
    cytokines = ["PBS"] + sorted(c for c in obs["cytokine"].astype(str).unique() if c != "PBS")
    obs["donor"] = pd.Categorical(obs["donor"].astype(str), categories=DONORS)
    obs["cytokine"] = pd.Categorical(obs["cytokine"].astype(str), categories=cytokines)
    obs["cell_type"] = pd.Categorical(obs["cell_type"].astype(str), categories=CELL_TYPES)
    obs["treatment"] = pd.Categorical(np.where(obs["cytokine"] == "PBS", "control", "treated"), categories=["control", "treated"])
    obs["psbulk_n_cells"] = obs["psbulk_n_cells"].astype(int)
    obs.index = (obs["donor"].astype(str) + "|" + obs["cell_type"].astype(str) + "|" + obs["cytokine"].astype(str)).values
    assert obs.index.is_unique
    # 3) the matrix: raw counts, as a plain dense table of numbers
    X = raw.X.toarray() if hasattr(raw.X, "toarray") else np.asarray(raw.X)
    X = np.ascontiguousarray(X, dtype=np.float32)
    var = pd.DataFrame(index=pd.Index(raw.var_names.astype(str), name=None))
    del raw
    # 4) our own columns: library size (obs) and total counts / detection (var)
    obs["library_size"] = X.sum(axis=1, dtype=np.float64).astype(np.int64)
    var["total_counts"] = X.sum(axis=0, dtype=np.float64).astype(np.int64)
    var["n_samples_detected"] = (X > 0).sum(axis=0).astype(int)
    adata = ad.AnnData(X=X, obs=obs, var=var)
    # 5) normalised versions of the matrix, kept in `layers` (more matrices with the same shape as X)
    lib = obs["library_size"].values.astype(np.float64)[:, None]
    cpm = (X / lib * 1e6).astype(np.float32)
    adata.layers["cpm"] = cpm
    adata.layers["log2cpm"] = np.log2(cpm + 1).astype(np.float32)
    return adata


# ── PCA ────────────────────────────────────────────────────────────────────────────────────────────
def pca_svd(M, n_components=10):
    """PCA of a samples × genes matrix: centre → SVD → project. Returns scores, % of variance explained, loadings (genes × PCs)."""
    Mc = M.astype(np.float64) - M.mean(axis=0)                    # 1) centre every gene
    U, S, Vt = np.linalg.svd(Mc, full_matrices=False)             # 2) SVD
    for k in range(min(n_components, len(S))):                    # the sign of a PC is arbitrary: make it reproducible
        j = np.argmax(np.abs(Vt[k]))
        if Vt[k, j] < 0:
            Vt[k], U[:, k] = -Vt[k], -U[:, k]
    scores = (U * S)[:, :n_components]                            # 3) projection: coordinates of the samples on the PCs
    var_pct = (S ** 2 / np.sum(S ** 2) * 100)[:n_components]
    return scores, var_pct, Vt[:n_components].T


def r2_categorical(y, groups):
    """Fraction of the variance of y explained by group membership (one-way ANOVA R²)."""
    y, g = np.asarray(y, float), np.asarray(groups).astype(str)
    ss_tot = ((y - y.mean()) ** 2).sum()
    codes, inv = np.unique(g, return_inverse=True)
    sums, counts = np.bincount(inv, weights=y), np.bincount(inv)
    return float((counts * (sums / counts - y.mean()) ** 2).sum() / ss_tot)


def pc_r2_table(scores, obs, n_pc=10):
    """R² of every PC with every sample variable (categories: ANOVA R²; numeric: squared Pearson correlation)."""
    loglib = np.log10(obs["library_size"].astype(float).values)
    logn = np.log10(obs["psbulk_n_cells"].astype(float).values)
    cols = {
        'tipo celular': lambda y: r2_categorical(y, obs["cell_type"]),
        "donor": lambda y: r2_categorical(y, obs["donor"]),
        "cytokine": lambda y: r2_categorical(y, obs["cytokine"]),
        'control vs tratado': lambda y: r2_categorical(y, obs["treatment"]),
        'tamaño de librería': lambda y: np.corrcoef(y, loglib)[0, 1] ** 2,
        'número de células': lambda y: np.corrcoef(y, logn)[0, 1] ** 2,
    }
    n_pc = min(n_pc, scores.shape[1])
    return pd.DataFrame({k: [f(scores[:, i]) if np.std(scores[:, i]) > 0 else 0.0 for i in range(n_pc)] for k, f in cols.items()},
                        index=[f"PC{i + 1}" for i in range(n_pc)])


def make_pca(adata, mask=None, n_hvg=N_HVG, expressed=None):
    """PCA on the `n_hvg` most variable genes (log2 CPM + 1) of the selected samples. Returns a dict with everything the plots need."""
    mask = np.ones(adata.n_obs, bool) if mask is None else np.asarray(mask)
    L = adata.layers["log2cpm"][mask]
    ok = np.ones(adata.n_vars, bool) if expressed is None else np.asarray(expressed)
    cand = np.flatnonzero(ok)
    var_g = L[:, cand].var(axis=0)
    hvg = cand[np.argsort(var_g)[::-1][:n_hvg]]
    scores, var_pct, loadings = pca_svd(L[:, hvg])
    obs = adata.obs[mask]
    return dict(mask=mask, obs=obs, scores=scores, var=var_pct, loadings=loadings, hvg=hvg,
                hvg_names=adata.var_names[hvg].to_numpy(), r2=pc_r2_table(scores, obs))


def ensure_pydeseq2():
    """pydeseq2 is only needed if the pre-computed results are missing: install it the first time it is needed."""
    import importlib.util, subprocess
    if importlib.util.find_spec("pydeseq2") is None:
        print('📦 Instalando pydeseq2 …')
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pydeseq2>=0.5.4"])
        importlib.invalidate_caches()


# ── differential expression (DESeq2) ──────────────────────────────────────────────────────────────────
def de_gene_universe(adata, expressed=None):
    """The genes that take part in the DE exercises: the immune list ∩ the genes expressed in the whole data set (≈ 500)."""
    expressed = is_expressed(adata.X) if expressed is None else expressed
    return [g for g in adata.var_names[expressed] if g in set(IMMUNE_GENES)]


def run_deseq2(adata, cell_type, treated, reference="PBS", n_cpus=1, genes=None):
    """DESeq2 (pydeseq2): `treated` vs `reference` in one cell type, 12 donors, design `~ donor + cytokine`, restricted to `genes`
    (default: the immune genes, see de_gene_universe). One row per gene of `adata` (NaN for the genes that were not tested).
    This is what produced the cached results."""
    genes = de_gene_universe(adata) if genes is None else genes
    ensure_pydeseq2()
    from pydeseq2.dds import DeseqDataSet
    from pydeseq2.ds import DeseqStats
    names = [f"{d}|{cell_type}|{c}" for c in (reference, treated) for d in DONORS]          # 12 reference + 12 treated samples
    rows = adata.obs_names.get_indexer(names)
    counts = pd.DataFrame(np.rint(adata.X[rows][:, adata.var_names.get_indexer(genes)]).astype(int), index=names, columns=genes)   # RAW counts, integers
    counts = counts.loc[:, is_expressed(counts.values)]                                      # expression filter on these 24 samples
    meta = pd.DataFrame({"donor": DONORS * 2, "cytokine": [reference] * 12 + [treated] * 12}, index=names)
    dds = DeseqDataSet(counts=counts, metadata=meta, design="~ donor + cytokine", quiet=True, n_cpus=n_cpus)
    dds.deseq2()                                                                             # size factors → dispersions → model fit
    stat = DeseqStats(dds, contrast=["cytokine", treated, reference], quiet=True, n_cpus=n_cpus)
    stat.summary()                                                                           # Wald test + Benjamini-Hochberg
    res = stat.results_df.rename(columns={"log2FoldChange": "log2fc", "lfcSE": "lfc_se", "pvalue": "pval", "baseMean": "base_mean"})
    return res[["base_mean", "log2fc", "lfc_se", "pval", "padj"]].reindex(adata.var_names)


CACHE_FIELDS = ("lfc", "nlp_adj")


def de_to_arrays(res):
    """Compact (float16) version of a DESeq2 table, for the cache: log2 fold change and -log10(adjusted p-value)."""
    f16 = lambda v: np.asarray(v, dtype=np.float64).astype(np.float16)
    return dict(lfc=f16(res["log2fc"].clip(-60, 60)), nlp_adj=f16(-np.log10(res["padj"].clip(lower=1e-300))))


class DEResults:
    """All the differential-expression results (cytokine vs PBS, per cell type), pre-computed once with DESeq2.
    `get(cell_type, cytokine)` returns a table with one row per gene; if the cache file is missing, it is computed on demand."""

    def __init__(self, adata, cache_path=None, verbose=True):
        self.adata, self.live, self._memo = adata, {}, {}
        self.arrays = None
        if cache_path and os.path.exists(cache_path):
            z = np.load(cache_path, allow_pickle=False)
            genes = pd.Index(z["genes"].astype(str))
            take = genes.get_indexer(adata.var_names)                              # re-order the genes like `adata`
            self.cell_types = [str(c) for c in z["cell_types"]]
            self.cytokines = [str(c) for c in z["cytokines"]]
            self.arrays = {k: self._reindex(z[k], take) for k in CACHE_FIELDS}
            self.meta = json.loads(str(z["meta"])) if "meta" in z.files else {}
            if verbose:
                print(f"✅ Resultados de DESeq2 precalculados: {len(self.cell_types)} tipos celulares × {len(self.cytokines)} citocinas")
        else:
            self.cell_types, self.cytokines, self.meta = list(CELL_TYPES), [c for c in adata.obs["cytokine"].cat.categories if c != "PBS"], {}

    @staticmethod
    def _reindex(a, take):
        out = np.full(a.shape[:2] + (len(take),), np.nan, dtype=np.float16)
        ok = take >= 0
        out[..., ok] = a[..., take[ok]]
        return out

    @property
    def cached(self):
        return self.arrays is not None

    def get(self, cell_type, cytokine):
        key = (cell_type, cytokine)
        if key in self._memo:
            return self._memo[key]
        if self.cached and cell_type in self.cell_types and cytokine in self.cytokines:
            i, j = self.cell_types.index(cell_type), self.cytokines.index(cytokine)
            nlp_adj = self.arrays["nlp_adj"][i, j].astype(np.float64)
            res = pd.DataFrame({"log2fc": self.arrays["lfc"][i, j].astype(np.float32), "padj": 10.0 ** -nlp_adj,
                                "neglog10_padj": nlp_adj}, index=self.adata.var_names)
        else:                                                                       # not in the cache: run DESeq2 now (≈ 15 s)
            res = run_deseq2(self.adata, cell_type, cytokine, n_cpus=2)[["log2fc", "padj"]].copy()
            res["neglog10_padj"] = -np.log10(res["padj"].clip(lower=1e-300))
        res["tested"] = res["padj"].notna()
        self._memo[key] = res
        return res

    def lfc_matrix(self, cell_type):
        """cytokines × genes matrix of log2 fold changes in one cell type (cache only)."""
        return self.arrays["lfc"][self.cell_types.index(cell_type)].astype(np.float32) if self.cached else None


def classify_de(res, padj_thresh=PADJ_DEFAULT, lfc_thresh=LFC_DEFAULT):
    """'up' / 'down' / 'ns' for every gene of a DE table (`ns` also for the genes that were not tested)."""
    sig = res["tested"] & (res["padj"] < padj_thresh)
    direction = pd.Series("ns", index=res.index)
    direction[sig & (res["log2fc"] > lfc_thresh)] = "up"
    direction[sig & (res["log2fc"] < -lfc_thresh)] = "down"
    return direction


# ══════════════════════════════════════════════════════════════════════════════════════════════════
#  20_widgets.py: the small toolbox every exercise is built from (controls, validation, layout, figures)
# ══════════════════════════════════════════════════════════════════════════════════════════════════
STRICT = False            # tests set this to True so that errors are raised instead of shown as a friendly message
FIG_SINK = None           # tests set this to a function(fig) that receives every figure shown (to save a PNG)
ERRORS = []               # every error caught while redrawing (the students see a friendly message; the tests read this list)

CSS = """
<style>
.cx-panel {border:1px solid rgba(127,127,127,.40); border-radius:8px; padding:8px 12px 6px 12px; background:rgba(127,127,127,.08); margin:2px 0 8px 0;}
.cx-title {font-weight:700; font-size:14px; margin:0 0 4px 0;}
.cx-label {font-weight:600; font-size:12px; margin:0 0 1px 2px;}
.cx-hint  {font-size:11px; opacity:.7; margin:0 0 2px 2px;}
.cx-warn  {border-left:4px solid #eda100; background:rgba(237,161,0,.14); padding:6px 10px; margin:4px 0; border-radius:4px; font-size:13px;}
.cx-err   {border-left:4px solid #e34948; background:rgba(227,73,72,.12); padding:6px 10px; margin:4px 0; border-radius:4px; font-size:13px;}
.cx-info  {border-left:4px solid #2a78d6; background:rgba(42,120,214,.10); padding:6px 10px; margin:4px 0; border-radius:4px; font-size:13px;}
.cx-code  {font-family:monospace; font-size:13px; line-height:1.45; white-space:pre; border:1px solid rgba(127,127,127,.35); border-radius:6px; padding:8px 12px; background:rgba(127,127,127,.07); overflow-x:auto;}
.cx-kw {color:#7a4fd6; font-weight:600} .cx-str {color:#1a8f5a} .cx-num {color:#d6571f} .cx-com {color:#8a8984; font-style:italic}
table.cx-table {border-collapse:collapse; font-size:12.5px; margin:2px 0;}
table.cx-table th {text-align:right; padding:3px 9px; border-bottom:2px solid rgba(127,127,127,.5); font-weight:600;}
table.cx-table td {text-align:right; padding:2px 9px; border-bottom:1px solid rgba(127,127,127,.2);}
table.cx-table td.l, table.cx-table th.l {text-align:left;}
table.cx-table tr:hover td {background:rgba(127,127,127,.12);}
.cx-up {color:#e34948; font-weight:600} .cx-down {color:#2a78d6; font-weight:600}
</style>
"""
STYLE = {"description_width": "initial"}


# ── messages shown in the output area (never a traceback) ───────────────────────────────────────────
def msg(text, kind="warn"):
    """Show a coloured message box. kind: 'warn' (orange), 'err' (red) or 'info' (blue)."""
    display(HTML(f'<div class="cx-{kind}">{text}</div>'))


# ── control builders ───────────────────────────────────────────────────────────────────────────────
def _lay(width=None, **kw):
    return widgets.Layout(width=width, **kw) if width else widgets.Layout(**kw)


def field(label, widget, hint=None):
    """A control with a small bold label above it (and an optional grey hint below)."""
    parts = [widgets.HTML(f'<div class="cx-label">{label}</div>'), widget]
    if hint:
        parts.append(widgets.HTML(f'<div class="cx-hint">{hint}</div>'))
    return widgets.VBox(parts, layout=_lay(margin="0 14px 4px 0"))


def dropdown(options, value=None, width="230px"):
    return widgets.Dropdown(options=options, value=value, layout=_lay(width), style=STYLE)


def toggle(options, value=None, width=None):
    return widgets.ToggleButtons(options=options, value=value, style={"button_width": width or "auto"})


def int_slider(lo, hi, value, step=1, width="260px"):
    return widgets.IntSlider(min=lo, max=hi, value=value, step=step, continuous_update=False, layout=_lay(width), style=STYLE)


def float_slider(lo, hi, value, step=0.1, width="260px", fmt=".2f"):
    return widgets.FloatSlider(min=lo, max=hi, value=value, step=step, continuous_update=False, readout_format=fmt, layout=_lay(width), style=STYLE)


def checkbox(label, value=False, width="260px"):
    return widgets.Checkbox(value=value, description=label, indent=False, layout=_lay(width))


def text_input(value="", width="320px", placeholder=""):
    return widgets.Text(value=value, placeholder=placeholder, continuous_update=False, layout=_lay(width), style=STYLE)


def multi_select(options, value, rows=6, width="200px"):
    return widgets.SelectMultiple(options=options, value=tuple(value), rows=rows, layout=_lay(width), style=STYLE)


def selection_summary(select, label_fn=lambda v: v, max_items=8):
    """A small line under a multi-select that spells out what is selected (the list only shows its first rows)."""
    box = widgets.HTML()

    def _update(*_):
        vals = list(select.value)
        shown = ", ".join(html.escape(str(label_fn(v))) for v in vals[:max_items]) + (f" … (+{len(vals) - max_items})" if len(vals) > max_items else "")
        box.value = f'<div class="cx-hint"><b>{len(vals)}</b> seleccionados: {shown if vals else "none"}</div>'
    select.observe(_update, "value")
    _update()
    return box


def gene_combo(value="ACTB", width="200px"):
    """One gene symbol with auto-completion (type a few letters, pick from the list, press Enter)."""
    return widgets.Combobox(value=value, options=list(GENE_NAMES), ensure_option=False, continuous_update=False,
                            placeholder='símbolo génico', layout=_lay(width), style=STYLE)


def geneset_dropdown(text_widget, default_label='— elige una lista predefinida —', width="260px"):
    """A dropdown of gene sets that, when used, writes the genes of the set into `text_widget`."""
    dd = dropdown([(default_label, "")] + [(k, k) for k in GENESETS], "", width)

    def _on(change):
        if change["new"]:
            text_widget.value = ", ".join(GENESETS[change["new"]])
            dd.value = ""                                         # back to the placeholder, so the same set can be chosen again
    dd.observe(_on, "value")
    return dd


def panel(title, *rows, note=None):
    """Titled grey box with rows of controls (each row = a widget or a list of widgets)."""
    kids = [widgets.HTML(f'<div class="cx-title">🎛 {title}</div>')]
    for r in rows:
        kids.append(widgets.HBox(list(r), layout=_lay(flex_flow="row wrap", align_items="flex-start")) if isinstance(r, (list, tuple)) else r)
    if note:
        kids.append(widgets.HTML(f'<div class="cx-hint">{note}</div>'))
    box = widgets.VBox(kids)
    box.add_class("cx-panel")
    return box


class Exercise:
    """Controls + a function that redraws the outputs every time a control changes (`widgets.interactive_output`).
    `render(**values)` receives the current value of every control (by name) and draws with matplotlib / display()."""

    def __init__(self, controls, render, ui_rows, title="Controls", note=None, extra=()):
        self.render, self.paused = render, False
        self.controls = dict(controls)
        self._tick = widgets.IntText(value=0)                         # (never displayed) bumping it redraws once, after several changes
        self.panel = panel(title, *ui_rows, note=note)
        self.output = widgets.interactive_output(self._call, {**self.controls, "_tick": self._tick})
        self.ui = widgets.VBox([widgets.HTML(CSS), self.panel, *extra, self.output])

    def values(self):
        return {k: w.value for k, w in self.controls.items()}

    def set_values(self, **values):
        """Change several controls and redraw ONCE."""
        self.paused = True
        try:
            for k, v in values.items():
                self.controls[k].value = v
        finally:
            self.paused = False
        self._tick.value += 1

    def _call(self, _tick=0, **kw):
        if self.paused:
            return
        try:
            self.render(**kw)
        except Exception as err:                                  # a student must never see a traceback
            ERRORS.append(f"{type(err).__name__}: {err}")
            if STRICT:
                raise
            msg(f"⚠️ Algo ha fallado con esta combinación ({html.escape(type(err).__name__)}: {html.escape(str(err))[:200]}). Prueba a cambiar un control o vuelve a los valores por defecto.", "err")

    def show(self):
        display(self.ui)
        return None


# ── figures ────────────────────────────────────────────────────────────────────────────────────────
def show_fig(fig):
    """Show a figure in the current output and free its memory."""
    if FIG_SINK is not None:
        FIG_SINK(fig)
    plt.show()
    plt.close(fig)


def fmt_n(x):
    return f"{x:,.0f}"


def si(x):
    """12,345,678 → '12.3 M'"""
    x = float(x)
    for lim, s in ((1e9, "G"), (1e6, "M"), (1e3, "k")):
        if abs(x) >= lim:
            return f"{x / lim:.1f} {s}"
    return f"{x:.0f}"


def html_table(df, index=True, float_fmt="{:,.2f}", classes="cx-table", left_cols=(), index_label=None, max_rows=200):
    """Small, clean HTML table (works in Jupyter, Colab, VS Code)."""
    df = df.head(max_rows)
    cols = list(df.columns)
    head = "".join(f'<th class="{"l" if c in left_cols else ""}">{html.escape(str(c))}</th>' for c in cols)
    if index:
        head = f'<th class="l">{html.escape(index_label or "")}</th>' + head
    rows = []
    for idx, r in df.iterrows():
        cells = []
        for c in cols:
            v = r[c]
            if isinstance(v, (float, np.floating)):
                s = "—" if not np.isfinite(v) else (float_fmt.format(v))
            elif isinstance(v, (int, np.integer)):
                s = f"{int(v):,}"
            else:
                s = str(v) if str(v).startswith("<") else html.escape(str(v))          # allow our own <span> markup
            cells.append(f'<td class="{"l" if c in left_cols else ""}">{s}</td>')
        rows.append("<tr>" + (f'<td class="l"><b>{html.escape(str(idx))}</b></td>' if index else "") + "".join(cells) + "</tr>")
    return f'<table class="{classes}"><thead><tr>{head}</tr></thead><tbody>{"".join(rows)}</tbody></table>'


def show_table(df, **kw):
    display(HTML(html_table(df, **kw)))


# ── genes: parsing and validation (names are case-insensitive; unknown ones get a friendly hint) ─────────────
GENE_NAMES, GENE_LOOKUP, GENESETS = [], {}, {}


def parse_genes(text, max_genes=60):
    """'stat1, Ifit3  nonsense' → (['STAT1', 'IFIT3'], ['nonsense'])"""
    valid, unknown = [], []
    for tok in [t for t in re.split(r"[,\s;]+", str(text).strip()) if t]:
        name = GENE_LOOKUP.get(tok.upper())
        if name is None:
            unknown.append(tok)
        elif name not in valid:
            valid.append(name)
    return valid[:max_genes], unknown


def unknown_genes_message(unknown):
    if not unknown:
        return
    parts = []
    for t in unknown[:6]:
        close = difflib.get_close_matches(t.upper(), list(GENE_LOOKUP), n=3, cutoff=0.7)
        parts.append(f"<b>{html.escape(t)}</b>" + (f" (¿quizá {', '.join(GENE_LOOKUP[c] for c in close)}?)" if close else ""))
    msg('Gen no encontrado en estos datos: ' + "; ".join(parts) + '. Revisa la ortografía (símbolos génicos, p. ej. <i>STAT1</i>).', "warn")


def one_gene(text, default=None):
    """Validate a single gene symbol; shows a hint and returns `default` (possibly None) if it is not valid."""
    valid, unknown = parse_genes(text, 1)
    unknown_genes_message(unknown if not valid else [])
    return valid[0] if valid else default


def gene_index(gene):
    return adata.var_names.get_loc(gene)


def sample_mask(cell_types=None, cytokines=None, donors=None):
    """Boolean mask over the samples of `adata` (None = no restriction)."""
    m = np.ones(adata.n_obs, bool)
    if cell_types is not None:
        m &= adata.obs["cell_type"].isin(list(cell_types)).values
    if cytokines is not None:
        m &= adata.obs["cytokine"].isin(list(cytokines)).values
    if donors is not None:
        m &= adata.obs["donor"].isin(list(donors)).values
    return m


def cyt_label(c):
    """Cytokine names as they are written in the data (IFN-gamma, IL-1-beta) → nicer for titles: IFN-γ, IL-1β."""
    c = re.sub(r"(IL-\d+)-(alpha|beta)", lambda m: m.group(1) + {"alpha": "α", "beta": "β"}[m.group(2)], str(c))        # IL-1-beta → IL-1β
    return c.replace("-gamma", "-γ").replace("-beta", "-β").replace("-alpha", "-α").replace("-omega", "-ω").replace("-lambda", "-λ").replace("-epsilon", "-ε")


def pick_default(options, wanted, fallback_n=3):
    """The wanted values that exist in `options` (so a default never breaks if the data change)."""
    ok = [w for w in wanted if w in options]
    return ok or list(options)[:fallback_n]


# ══════════════════════════════════════════════════════════════════════════════════════════════════
#  Exercise 1: variables and print()
# ══════════════════════════════════════════════════════════════════════════════════════════════════
def highlight_python(code):
    """Tiny syntax highlighter (strings, numbers, comments, keywords) → HTML for the code panel."""
    out = []
    pattern = re.compile(r"(#.*$)|(f?'[^']*'|f?\"[^\"]*\")|(\b\d+(?:\.\d+)?\b)|(\b(?:print|len|True|False)\b)")
    for line in html.escape(code, quote=False).split("\n"):
        def sub(m):
            for cls, g in zip(("com", "str", "num", "kw"), m.groups()):
                if g:
                    return f'<span class="cx-{cls}">{g}</span>'
        out.append(pattern.sub(sub, line))
    return "\n".join(out)


def ex1_program(message, n_donors, cytokine, genes_text, is_control, n_cells, umis_per_cell):
    """The Python program that corresponds to the current values of the controls (as text)."""
    genes = [g.strip() for g in re.split(r"[,;]", genes_text) if g.strip()]
    return "\n".join([
        '# 1) Crea variables: un NOMBRE a la izquierda de  =  y un VALOR a la derecha',
        f"message = {message!r}",
        f"n_donors = {int(n_donors)}",
        f"cytokine = {cytokine!r}",
        f"genes = {genes!r}",
        f"is_control = {bool(is_control)}",
        f"n_cells = {int(n_cells)}",
        f"umis_per_cell = {float(umis_per_cell)!r}",
        "",
        '# 2) Úsalas: print() muestra un valor en pantalla',
        "print(message)",
        "print('Donantes:', n_donors)",
        "print(cytokine, 'se añadió a', n_donors, 'donantes')",
        "print('Número de genes:', len(genes))",
        "",
        '# 3) Las variables se pueden combinar para crear otras nuevas',
        "library_size = n_cells * umis_per_cell",
        "print('Tamaño de librería:', library_size)",
        "print('¿Es esta muestra un control?', is_control)",
    ])


def ex1_run(program):
    """Run the (safe: built by ex1_program from repr() literals) program; returns what it printed and its variables."""
    buf, ns = io.StringIO(), {}
    real_stdout = sys.stdout
    sys.stdout = buf
    try:
        exec(program, {"__builtins__": {"print": print, "len": len}}, ns)
    finally:
        sys.stdout = real_stdout
    return buf.getvalue(), ns


def ex1_render(message, n_donors, cytokine, genes_text, is_control, n_cells, umis_per_cell):
    program = ex1_program(message, n_donors, cytokine, genes_text, is_control, n_cells, umis_per_cell)
    printed, ns = ex1_run(program)
    badge = {"str": "#1a8f5a", "int": "#d6571f", "float": "#d6571f", "bool": "#7a4fd6", "list": "#2a78d6"}
    rows = "".join(
        f'<tr><td class="l"><b>{k}</b></td><td class="l">{html.escape(repr(v))}</td>'
        f'<td class="l"><span style="color:{badge.get(type(v).__name__, "#888")};font-weight:600">{type(v).__name__}</span></td></tr>'
        for k, v in ns.items())
    table = f'<table class=\"cx-table\"><thead><tr><th class=\"l\">nombre</th><th class=\"l\">valor guardado</th><th class=\"l\">tipo</th></tr></thead><tbody>{rows}</tbody></table>'
    display(HTML(
        f'''<div style=\"display:flex; flex-wrap:wrap; gap:16px; align-items:flex-start\"><div style=\"flex:1 1 440px\"><div class=\"cx-label\">🐍 El código Python escrito por tus controles</div><div class=\"cx-code\">{highlight_python(program)}</div></div><div style=\"flex:1 1 320px\"><div class=\"cx-label\">🖨️ Lo que muestra <code>print()</code></div><div class=\"cx-code\" style=\"background:rgba(42,120,214,.10)\">{html.escape(printed)}</div><div class=\"cx-label\" style=\"margin-top:10px\">🗃️ Lo que Python recuerda (sus variables)</div>{table}</div></div>'''))


def exercise_variables():
    c = dict(
        message=text_input('¡Hola, inmunólogos!', "300px"),
        n_donors=int_slider(1, 12, 12, width="240px"),
        cytokine=dropdown(["IFN-gamma", "IL-4", "IL-6", "TNF-alpha", "IL-2", "IL-10"], "IFN-gamma", "170px"),
        genes_text=text_input("STAT1, CXCL9, ACTB", "250px"),
        is_control=checkbox('esta muestra es un control', False, "210px"),
        n_cells=int_slider(100, 5000, 2500, 100, "240px"),
        umis_per_cell=float_slider(1000, 8000, 3000, 250, "240px", ".0f"),
    )
    rows = [[field('message  (texto)', c["message"], 'cualquier texto'), field('cytokine  (texto)', c["cytokine"]),
             field('genes  (una lista de textos)', c["genes_text"], 'separa con comas')],
            [field('n_donors  (número entero)', c["n_donors"]), field('n_cells  (número entero)', c["n_cells"]),
             field('umis_per_cell  (número decimal)', c["umis_per_cell"])],
            [field('is_control  (True / False)', c["is_control"])]]
    ex = Exercise(c, ex1_render, rows, title='Crea tus variables')
    ex.show()
    return ex


# ══════════════════════════════════════════════════════════════════════════════════════════════════
#  Exercise 2: basic plots (simulated data, independent of the dataset)
# ══════════════════════════════════════════════════════════════════════════════════════════════════
PLOT_TYPES = [('📊 Histograma', "hist"), ('📦 Diagrama de cajas', "box"), ('📶 Gráfico de barras', "bar"), ('🔵 Diagrama de dispersión', "scatter"), ('📈 Gráfico de líneas (curso temporal)', "line")]
COL_CTRL, COL_TRT = "#8a8984", "#eb6834"


def sim_expression(n, mean, effect, sd, r, seed):
    """Fake expression values (log2 CPM) of ONE gene in `n` donors: without (control) and with (treated) a stimulus.
    `effect` = log2 fold change; `r` = how much a donor's two values resemble each other (same person, same baseline)."""
    rng = np.random.default_rng(int(seed))
    z1 = rng.standard_normal(n)
    z2 = r * z1 + np.sqrt(max(1 - r ** 2, 0)) * rng.standard_normal(n)
    return mean + sd * z1, mean + effect + sd * z2


def ex2_render(plot_type, n, mean, effect, sd, bins, r, tau, seed):
    control, treated = sim_expression(n, mean, effect, sd, r, seed)
    fig, ax = plt.subplots(figsize=(7.4, 4.3))
    if plot_type == "hist":
        lo, hi = min(control.min(), treated.min()), max(control.max(), treated.max())
        edges = np.linspace(lo, hi, bins + 1)
        ax.hist(control, bins=edges, color=COL_CTRL, alpha=0.65, label=disp("control"))
        ax.hist(treated, bins=edges, color=COL_TRT, alpha=0.65, label=disp("treated"))
        ax.axvline(control.mean(), color=COL_CTRL, lw=2, ls="--"); ax.axvline(treated.mean(), color=COL_TRT, lw=2, ls="--")
        ax.set_xlabel('expresión (log2 CPM)'); ax.set_ylabel('número de donantes'); ax.legend()
        ax.set_title(f"Histograma: ¿cuántos donantes tienen cada valor? (discontinua = media)")
        code = f"plt.hist(tratado, bins={bins})"
    elif plot_type == "box":
        bp = ax.boxplot([control, treated], widths=0.5, patch_artist=True, showfliers=False, medianprops=dict(color="k", lw=2))
        for patch, col in zip(bp["boxes"], (COL_CTRL, COL_TRT)):
            patch.set(facecolor=col, alpha=0.45)
        rng = np.random.default_rng(1)
        for i, v in enumerate((control, treated), 1):
            ax.scatter(i + rng.uniform(-0.15, 0.15, len(v)), v, s=14, color=(COL_CTRL, COL_TRT)[i - 1], alpha=0.8, zorder=3)
        ax.set_xticks([1, 2]); ax.set_xticklabels([disp("control"), disp("treated")]); ax.set_ylabel('expresión (log2 CPM)')
        ax.set_title('Diagrama de cajas: mediana (línea), 50 % central (caja), rango (bigotes)')
        code = 'plt.boxplot([control, tratado])'
    elif plot_type == "bar":
        means, sds = [control.mean(), treated.mean()], [control.std(ddof=1), treated.std(ddof=1)]
        ax.bar([0, 1], means, yerr=sds, capsize=8, color=(COL_CTRL, COL_TRT), alpha=0.8, width=0.55)
        ax.set_xticks([0, 1]); ax.set_xticklabels([disp("control"), disp("treated")]); ax.set_ylabel('expresión (log2 CPM): media ± DE')
        ax.set_title('Barras: un número (la media) por grupo')
        code = "plt.bar(['control', 'tratado'], [control.mean(), tratado.mean()])"
    elif plot_type == "scatter":
        ax.scatter(control, treated, s=26, color=BLUE, alpha=0.75, edgecolor="white", linewidth=0.5)
        lo, hi = min(control.min(), treated.min()) - 0.5, max(control.max(), treated.max()) + 0.5
        ax.plot([lo, hi], [lo, hi], color="k", lw=1, ls="--"); ax.text(hi, hi, " y = x", va="top", ha="right", fontsize=8)
        ax.set_xlim(lo, hi); ax.set_ylim(lo, hi); ax.set_aspect("equal")
        rr = np.corrcoef(control, treated)[0, 1] if n > 2 else float("nan")
        ax.set_xlabel('control (log2 CPM)'); ax.set_ylabel('tratado (log2 CPM)')
        ax.set_title(f"Dispersión: un punto por donante (correlación r = {rr:.2f})")
        code = 'plt.scatter(control, tratado)'
    else:                                                          # line plot: a time course
        t = np.linspace(0, 24, 13)
        rng = np.random.default_rng(int(seed) + 1000)
        k = min(int(n), 12)                                         # at most 12 donors as thin lines
        for i in range(k):
            ax.plot(t, mean + (control[i] - mean) + effect * (1 - np.exp(-t / tau)) + rng.normal(0, 0.12 * sd, len(t)), color=COL_TRT, alpha=0.25, lw=1)
        ax.plot(t, mean + effect * (1 - np.exp(-t / tau)), color=COL_TRT, lw=3, label='respuesta media (tratado)')
        ax.axhline(mean, color=COL_CTRL, lw=2, ls="--", label='control (sin cambio)')
        ax.set_xlabel('horas tras añadir la citocina'); ax.set_ylabel('expresión (log2 CPM)'); ax.legend(loc="best")
        ax.set_title(f"Líneas: expresión a lo largo del tiempo ({k} donantes, líneas finas)")
        code = 'plt.plot(horas, expresion)'
    fig.tight_layout()
    show_fig(fig)
    display(HTML(f'<div class=\"cx-hint\">Detrás del gráfico, Python ejecuta una sola línea:</div><div class=\"cx-code\">{html.escape(code)}</div>'))


def exercise_plotting():
    c = dict(plot_type=dropdown(PLOT_TYPES, "hist", "230px"), n=int_slider(5, 200, 60), mean=float_slider(0, 10, 5, 0.5, fmt=".1f"),
             effect=float_slider(-3, 3, 1.0, 0.25, fmt="+.2f"), sd=float_slider(0.1, 3, 1.0, 0.1, fmt=".1f"), bins=int_slider(3, 60, 20),
             r=float_slider(0, 0.99, 0.7, 0.05), tau=float_slider(1, 12, 4, 1, fmt=".0f"), seed=widgets.IntText(value=1, layout=_lay("70px")))
    dice = widgets.Button(description='🎲 nuevo sorteo aleatorio', layout=_lay("170px"))
    dice.on_click(lambda _b: setattr(c["seed"], "value", c["seed"].value + 1))
    f_bins = field('bins  (histograma)', c["bins"], 'cuántas barras')
    f_r = field('parecido entre los dos valores de un donante  (dispersión)', c["r"], '0 = sin relación, 1 = mismo patrón')
    f_tau = field('velocidad de respuesta: horas hasta el 63 %  (líneas)', c["tau"])
    rows = [[field('tipo de gráfico', c["plot_type"]), field('número de donantes', c["n"]), field('expresión media (control)', c["mean"])],
            [field('efecto de la citocina (log2 fold change)', c["effect"], '+1 = el doble, −1 = la mitad'), field('variabilidad entre donantes (DE)', c["sd"]),
             f_bins, f_r, f_tau],
            [field('número del sorteo', c["seed"]), widgets.VBox([widgets.HTML('<div class="cx-label">&nbsp;</div>'), dice])]]
    show_for = {"hist": [f_bins], "scatter": [f_r], "line": [f_tau]}

    def _toggle(*_):
        for f in (f_bins, f_r, f_tau):
            f.layout.display = "none"
        for f in show_for.get(c["plot_type"].value, []):
            f.layout.display = None
    c["plot_type"].observe(_toggle, "value")
    _toggle()
    ex = Exercise(c, ex2_render, rows, title='Simula un gen y elige cómo mirarlo',
                  note='Los datos son <b>simulados</b> (todavía no usamos datos reales): un gen medido en varios donantes, sin y con una citocina.')
    ex.show()
    return ex


# ══════════════════════════════════════════════════════════════════════════════════════════════════
#  Exercise 3: exploring the AnnData object (X, obs, var) and subsetting it
# ══════════════════════════════════════════════════════════════════════════════════════════════════
class SamplePicker:
    """Pick samples from the 4,224 sample IDs: three dropdowns narrow the list, and the selection survives the filtering."""
    ALL = '(todos)'

    def __init__(self, default, rows=9, width="400px", max_options=400):
        obs = adata.obs
        order = np.lexsort((obs["cytokine"].cat.codes.values, obs["cell_type"].cat.codes.values, obs["donor"].cat.codes.values))
        self.ids = list(obs.index[order])                                       # by donor, then cell type, then cytokine
        self.rank = {s: i for i, s in enumerate(self.ids)}
        self.pos = adata.obs_names.get_indexer(self.ids)
        self.max_options = max_options
        self.chosen = [s for s in default if s in self.rank]
        self.f_donor = dropdown([self.ALL] + DONORS, self.ALL, "130px")
        self.f_ct = dropdown([self.ALL] + CELL_TYPES, self.ALL, "150px")
        self.f_cyt = dropdown([self.ALL] + list(obs["cytokine"].cat.categories), self.ALL, "150px")
        self.select = multi_select([], [], rows=rows, width=width)
        self.value_widget = widgets.Text(value="\n".join(self.chosen))             # (never displayed) the selection, as ONE value that only
        self._busy = False                                                          # changes when the student selects, not when the list is filtered
        self.status = widgets.HTML()
        for f in (self.f_donor, self.f_ct, self.f_cyt):
            f.observe(self._refresh, "value")
        self.select.observe(self._on_select, "value")
        self._refresh()
        self.widget = widgets.VBox([
            widgets.HBox([field("Donor", self.f_donor), field('Tipo celular', self.f_ct), field("Cytokine", self.f_cyt)]),
            self.select, self.status])

    def _refresh(self, *_):
        obs = adata.obs
        m = np.ones(adata.n_obs, bool)
        if self.f_donor.value != self.ALL:
            m &= (obs["donor"] == self.f_donor.value).values
        if self.f_ct.value != self.ALL:
            m &= (obs["cell_type"] == self.f_ct.value).values
        if self.f_cyt.value != self.ALL:
            m &= (obs["cytokine"] == self.f_cyt.value).values
        matches = [s for s, p in zip(self.ids, self.pos) if m[p]]
        shown = matches[:self.max_options]
        opts = sorted(set(shown) | set(self.chosen), key=self.rank.get)
        self._busy = True
        try:
            self.select.value = ()
            self.select.options = opts
            self.select.value = tuple(s for s in self.chosen if s in opts)
        finally:
            self._busy = False
        extra = f" (se muestran las primeras {self.max_options}: acótalo con los filtros)" if len(matches) > self.max_options else ""
        self._info = f"{len(matches):,} muestras cumplen los filtros{extra}. Ctrl/⌘ + clic selecciona varias."
        self._update_status()

    def _update_status(self):
        self.status.value = f'<div class="cx-hint">{self._info} <b>{len(self.chosen)}</b> seleccionadas.</div>'

    def _on_select(self, change):
        if self._busy:
            return
        self.chosen = list(change["new"])
        self.value_widget.value = "\n".join(self.chosen)
        self._update_status()


def gene_picker(default, width="330px", placeholder='STAT1, CXCL9, ...', with_sets=True, label='Genes (símbolos, separados por comas)'):
    """Genes as a list of symbols (editable text) + a ready-made-lists dropdown + an auto-complete box to add one gene."""
    text = text_input(default, width, placeholder)
    sets = geneset_dropdown(text) if with_sets else None
    add = widgets.Combobox(value="", options=list(GENE_NAMES), ensure_option=False, continuous_update=False,
                           placeholder='escribe un gen y pulsa Enter', layout=_lay("190px"), style=STYLE)

    def _add(change):
        new = change["new"]
        if new:
            g, _unknown = parse_genes(new, 1)
            if g and g[0] not in parse_genes(text.value, 500)[0]:
                text.value = (text.value.strip().rstrip(",") + ", " + g[0]).lstrip(", ")
            add.value = ""
    add.observe(_add, "value")
    second = [field('…o una lista predefinida', sets)] if with_sets else []
    return text, widgets.VBox([field(label, text), widgets.HBox(second + [field('…o añade un gen' if with_sets else 'Añade un gen (autocompletar)', add)])])


MATRICES = [('X: conteos crudos', "X"), ("capa 'cpm': conteos por millón", "cpm"), ("capa 'log2cpm': log2(CPM + 1)", "log2cpm")]


def anndata_repr(a):
    """The classic text of an AnnData object (we write it ourselves so that it looks the same with every anndata version)."""
    lines = [f"AnnData object with n_obs × n_vars = {a.n_obs} × {a.n_vars}",
             "    obs: " + ", ".join(f"'{c}'" for c in a.obs.columns),
             "    var: " + ", ".join(f"'{c}'" for c in a.var.columns)]
    layers = [k for k in a.layers.keys() if k is not None]
    if layers:
        lines.append("    layers: " + ", ".join(f"'{k}'" for k in layers))
    return "\n".join(lines)


def _anndata_cartoon(sub):
    """Left: the AnnData scheme with the shapes of the selection. Right: where the selection sits in the full matrix."""
    n, m = sub.n_obs, sub.n_vars
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10.5, 3.3), gridspec_kw={"width_ratios": [1.55, 1]})
    a1.set_xlim(0, 10); a1.set_ylim(0, 5.6); a1.axis("off")
    box = lambda x, y, w, h, col, txt, tc="white": (a1.add_patch(plt.Rectangle((x, y), w, h, fc=col, ec="white", lw=2)),
                                                     a1.text(x + w / 2, y + h / 2, txt, ha="center", va="center", color=tc, fontsize=10, fontweight="bold"))
    box(2.5, 0.4, 5.2, 3.5, BLUE, f"X\n{n} muestras × {m} genes\n(los números)")
    box(0.1, 0.4, 2.2, 3.5, ORANGE, f"obs\n{n} muestras ×\n{sub.obs.shape[1]} columnas\n(sobre las\nMUESTRAS)")
    box(2.5, 4.1, 5.2, 1.3, GREEN, f"var\n{m} genes × {sub.var.shape[1]} columnas\n(sobre los GENES)")
    a1.text(8.0, 2.3, f"layers\n{', '.join(k for k in sub.layers.keys() if k is not None)}\n(misma forma que X)", va="center", fontsize=8.5, color=PBS_COLOR)
    a1.set_title('El objeto AnnData que has seleccionado', loc="left")
    # right: where the subset is in the complete matrix
    a2.set_xlim(0, adata.n_vars); a2.set_ylim(adata.n_obs, 0)
    a2.add_patch(plt.Rectangle((0, 0), adata.n_vars, adata.n_obs, fc="#eef3fb", ec=BLUE, lw=1.2))
    ri, ci = adata.obs_names.get_indexer(sub.obs_names), adata.var_names.get_indexer(sub.var_names)
    for r in ri:
        a2.axhline(r, color=ORANGE, lw=1.4, alpha=0.9)
    for c in ci:
        a2.axvline(c, color=GREEN, lw=1.4, alpha=0.9)
    a2.scatter(np.tile(ci, len(ri)), np.repeat(ri, len(ci)), s=14, color=BLUE, zorder=3)
    a2.set_xticks([]); a2.set_yticks([])
    a2.set_xlabel(f"{adata.n_vars:,} genes", fontsize=9); a2.set_ylabel(f"{adata.n_obs:,} muestras", fontsize=9)
    a2.spines[:].set_visible(False)
    a2.set_title('…dentro de la matriz completa', loc="left")
    fig.tight_layout()
    show_fig(fig)


def ex3_render(samples, genes_text, matrix):
    genes, unknown = parse_genes(genes_text, 200)
    unknown_genes_message(unknown)
    samples = [s for s in str(samples).split("\n") if s in adata.obs_names]
    if not samples or not genes:
        msg('Selecciona <b>al menos una muestra</b> y <b>un gen</b> para crear un subconjunto.', "info")
        return
    sub = adata[samples, genes]                                                    # ← the subset: samples first, genes second
    display(HTML(f'''<div class=\"cx-label\">El código que crea tu subconjunto</div><div class=\"cx-code\">sub = adata[samples, genes]      <span class=\"cx-com\"># {len(samples)} nombres de muestra, {len(genes)} nombres de gen</span>\nprint(sub)</div><div class=\"cx-code\" style=\"margin-top:4px;background:rgba(42,120,214,.10)\">{html.escape(anndata_repr(sub))}</div>'''))
    _anndata_cartoon(sub)
    M = sub.X if matrix == "X" else sub.layers[matrix]
    dfx = pd.DataFrame(np.asarray(M), index=sub.obs_names, columns=sub.var_names)
    fmt = "{:,.0f}" if matrix == "X" else ("{:,.1f}" if matrix == "cpm" else "{:.2f}")
    shade = np.log10(dfx.values.astype(float) + 1) if matrix != "log2cpm" else dfx.values.astype(float)
    top = max(float(np.nanmax(shade)), 1e-9) * 1.25
    head = "".join(f'<th>{html.escape(str(c))}</th>' for c in dfx.columns)                      # plain HTML: no jinja2 needed
    body = "".join(
        f'<tr><td class="l"><b>{html.escape(str(idx))}</b></td>' + "".join(
            f'<td style="background:rgba(42,120,214,{0.05 + 0.6 * max(0.0, float(sh)) / top:.2f})">{fmt.format(v)}</td>' for v, sh in zip(row, shrow)) + "</tr>"
        for idx, row, shrow in zip(dfx.index, dfx.values, shade))
    table_x = f'<table class="cx-table"><thead><tr><th class="l"></th>{head}</tr></thead><tbody>{body}</tbody></table>'
    obs_df = sub.obs.copy()
    for c in ("donor", "cytokine", "treatment", "cell_type"):
        obs_df[c] = obs_df[c].astype(str)
    obs_df["psbulk_n_cells"] = obs_df["psbulk_n_cells"].map("{:,}".format); obs_df["library_size"] = obs_df["library_size"].map("{:,}".format)
    var_df = sub.var.copy()
    var_df["total_counts"] = var_df["total_counts"].map("{:,}".format)
    tabs = widgets.Output(), widgets.Output(), widgets.Output()
    with tabs[0]:
        display(HTML(f'<div class=\"cx-hint\">Filas = las {len(samples)} muestras (las mismas filas que <code>obs</code>), columnas = los {len(genes)} genes (las mismas filas que <code>var</code>).</div>'))
        display(HTML(table_x))
    with tabs[1]:
        display(HTML('<div class="cx-hint">Una fila por muestra: de quién procede, qué se añadió, qué tipo celular, y cuántas células / conteos contiene.</div>'))
        display(HTML(html_table(obs_df, left_cols=obs_df.columns, index_label='ID de muestra')))
    with tabs[2]:
        display(HTML('<div class="cx-hint">Una fila por gen: sus conteos totales en todos los datos, y en cuántas de las 4.224 muestras tiene al menos un conteo.</div>'))
        display(HTML(html_table(var_df, left_cols=["total_counts"], index_label=disp("gene"))))
    tab = widgets.Tab(children=list(tabs))
    for i, t in enumerate(['X  (los números)', 'obs  (muestras)', 'var  (genes)']):
        tab.set_title(i, t)
    display(tab)


def exercise_anndata():
    default = [f"Donor1|{ct}|PBS" for ct in CELL_TYPES] + ["Donor1|CD14_Mono|IFN-gamma"]
    sp = SamplePicker(default)
    gtext, gwidget = gene_picker("LYZ, CD3E, NKG7, STAT1, ACTB")
    matrix = dropdown(MATRICES, "X", "300px")
    c = dict(samples=sp.value_widget, genes_text=gtext, matrix=matrix)
    rows = [widgets.HBox([widgets.VBox([widgets.HTML('<div class="cx-label">① Elige muestras (filas)</div>'), sp.widget]),
                          widgets.VBox([widgets.HTML('<div class="cx-label">② Elige genes (columnas)</div>'), gwidget,
                                        field('③ Qué matriz mirar', matrix, 'X contiene los conteos crudos; las capas, versiones normalizadas (Ejercicios 5-6)')],
                                       layout=_lay(margin="0 0 0 20px"))],
                         layout=_lay(flex_flow="row wrap"))]
    ex = Exercise(c, ex3_render, rows, title='Construye tu propio subconjunto de los datos')
    ex.picker = sp
    ex.show()
    return ex


# ══════════════════════════════════════════════════════════════════════════════════════════════════
#  Exercise 4: distributions of raw counts (boxplots of pseudobulk samples)
# ══════════════════════════════════════════════════════════════════════════════════════════════════
SPLITS = [('tipo celular', "cell_type"), ('control (PBS) vs tratado', "treatment"), (disp("donor"), "donor"), ('citocina (las seleccionadas abajo)', "cytokine")]
SPLIT_NAME = {"cell_type": 'tipo celular', "treatment": disp("condition"), "donor": disp("donor"), "cytokine": disp("cytokine")}


def group_colors(split_by, groups):
    if split_by == "cell_type":
        return {g: CT_COLORS[g] for g in groups}
    if split_by == "treatment":
        return {"control": "#8a8984", "treated": ORANGE}
    pal = DONOR_HUES if split_by == "donor" else CYT_HUES
    return {g: pal[i % len(pal)] for i, g in enumerate(groups)}


def group_label(split_by, g):
    return CT_LABEL.get(g, g) if split_by == "cell_type" else (cyt_label(g) if split_by == "cytokine" else disp(g))


def draw_boxes(ax, data, genes, groups, colors, log_y=True, points=True, label_fn=lambda g: g):
    """Boxes of `data[gene][group]` (arrays), grouped by gene on the x axis and coloured by group."""
    G = len(groups)
    w = 0.8 / G
    rng = np.random.default_rng(0)
    floor = 0.5
    for i, gene in enumerate(genes):
        for j, g in enumerate(groups):
            v = np.asarray(data[gene][g], float)
            if len(v) == 0:
                continue
            v = np.maximum(v, floor) if log_y else v
            pos = i + (j - (G - 1) / 2) * w
            bp = ax.boxplot([v], positions=[pos], widths=w * 0.85, patch_artist=True, showfliers=False, medianprops=dict(color="k", lw=1.6),
                            whiskerprops=dict(color="#666", lw=0.9), capprops=dict(color="#666", lw=0.9))
            bp["boxes"][0].set(facecolor=colors[g], alpha=0.55, edgecolor="#555", lw=0.9)
            if points:
                vv = v if len(v) <= 120 else rng.choice(v, 120, replace=False)
                ax.scatter(pos + rng.uniform(-w * 0.3, w * 0.3, len(vv)), vv, s=5, color=colors[g], alpha=0.55, linewidths=0, zorder=3)
    ax.set_xticks(range(len(genes))); ax.set_xticklabels(genes, fontsize=9.5, fontstyle="italic")
    ax.set_xlim(-0.6, len(genes) - 0.4)
    if log_y:
        ax.set_yscale("log")


def ex4_render(genes_text, cell_types, split_by, all_cytokines, cytokines, log_y, show_points):
    genes, unknown = parse_genes(genes_text, 12)
    unknown_genes_message(unknown)
    cell_types, cytokines = list(cell_types), list(cytokines)
    if not genes:
        msg('Escribe al menos un <b>símbolo génico</b> válido (por ejemplo <i>ACTB, LYZ, CD3E</i>).', "info"); return
    if not cell_types:
        msg('Selecciona al menos un <b>tipo celular</b>.', "info"); return
    if split_by == "cytokine" and not cytokines:
        msg('Para separar por citocina, selecciona al menos una en la lista (Ctrl/⌘ para varias).', "info"); return
    m = sample_mask(cell_types, None if (all_cytokines and split_by != "cytokine") else (cytokines or None))
    if m.sum() == 0:
        msg('Ninguna muestra cumple esta selección.', "warn"); return
    obs = adata.obs[m]
    if split_by == "cytokine":
        groups = [c for c in obs["cytokine"].cat.categories if c in cytokines][:10]
    else:
        groups = [g for g in obs[split_by].cat.categories if g in set(obs[split_by])]
    keep = obs[split_by].isin(groups).values
    rows = np.flatnonzero(m)[keep]
    labels = obs[split_by].astype(str).values[keep]
    colors = group_colors(split_by, groups)
    cols = [gene_index(g) for g in genes]
    block = adata.X[np.ix_(rows, cols)]
    data = {gn: {g: block[labels == g, k] for g in groups} for k, gn in enumerate(genes)}
    lib = {g: adata.obs["library_size"].values[rows][labels == g].astype(float) for g in groups}
    G, nG = len(groups), len(genes)
    fig, axes = plt.subplots(1, 2, figsize=(max(8.5, 1.15 * nG * max(G, 2) ** 0.5 + 4.2), 4.6), gridspec_kw={"width_ratios": [max(nG * (0.5 + 0.3 * G), 3), max(G, 2)]})
    draw_boxes(axes[0], data, genes, groups, colors, log_y, show_points)
    axes[0].set_ylabel('conteos crudos del gen en una muestra' + ('  (escala log)' if log_y else ""))
    axes[0].set_title('Cada caja = las muestras pseudobulk de un grupo')
    ax = axes[1]
    for j, g in enumerate(groups):
        v = lib[g]
        bp = ax.boxplot([v], positions=[j], widths=0.6, patch_artist=True, showfliers=False, medianprops=dict(color="k", lw=1.6))
        bp["boxes"][0].set(facecolor=colors[g], alpha=0.55, edgecolor="#555")
        if show_points:
            rng = np.random.default_rng(j)
            vv = v if len(v) <= 120 else rng.choice(v, 120, replace=False)
            ax.scatter(j + rng.uniform(-0.2, 0.2, len(vv)), vv, s=5, color=colors[g], alpha=0.55, linewidths=0, zorder=3)
    ax.set_xticks(range(G)); ax.set_xticklabels([group_label(split_by, g) for g in groups], rotation=35 if G > 2 else 0, ha="right" if G > 2 else "center", fontsize=8.5)
    ax.set_yscale("log" if log_y else "linear"); ax.set_ylabel('tamaño de librería (conteos totales de la muestra)'); ax.set_title('Conteos totales por muestra')
    handles = [Line2D([], [], marker="s", ls="", color=colors[g], alpha=0.7, label=f"{group_label(split_by, g)}  (n = {int((labels == g).sum())})") for g in groups]
    fig.legend(handles=handles, loc="lower center", ncol=min(G, 5), bbox_to_anchor=(0.5, -0.06), fontsize=8.5, title=f"color = {SPLIT_NAME[split_by]}", title_fontsize=9)
    fig.tight_layout()
    show_fig(fig)
    # the numbers behind the boxes
    tab = pd.DataFrame({group_label(split_by, g): [np.median(data[gn][g]) for gn in genes] + [np.median(lib[g])] for g in groups}, index=genes + ['tamaño de librería'])
    tab = tab.map(lambda x: f"{x:,.0f}") if hasattr(tab, "map") else tab.applymap(lambda x: f"{x:,.0f}")
    display(HTML('<div class="cx-label">Mediana de cada caja (conteos crudos)</div>' + html_table(tab, index_label=disp("gene"))))


def exercise_boxplots():
    gtext, gwidget = gene_picker("ACTB, GAPDH, LYZ, CD3E, NKG7, STAT1", placeholder="ACTB, LYZ, ...")
    cyts = list(adata.obs["cytokine"].cat.categories)
    c = dict(genes_text=gtext, cell_types=multi_select([(CT_LABEL[x], x) for x in CELL_TYPES], CELL_TYPES, rows=4, width="170px"),
             split_by=dropdown(SPLITS, "cell_type", "270px"), all_cytokines=checkbox('usar las 88 condiciones', True, "190px"),
             cytokines=multi_select(cyts, pick_default(cyts, ["PBS", "IFN-gamma", "IL-4", "IL-1-beta"]), rows=6, width="170px"),
             log_y=checkbox('escala log (eje y)', True, "170px"), show_points=checkbox('mostrar las muestras como puntos', True, "200px"))

    def _sync(*_):
        c["cytokines"].disabled = c["all_cytokines"].value and c["split_by"].value != "cytokine"
    c["all_cytokines"].observe(_sync, "value"); c["split_by"].observe(_sync, "value"); _sync()
    rows = [[widgets.VBox([gwidget]), field('Tipos celulares', c["cell_types"]), field('Agrupar las cajas por', c["split_by"]),
             widgets.VBox([field('Citocinas', c["cytokines"]), selection_summary(c["cytokines"], cyt_label), c["all_cytokines"]])], [c["log_y"], c["show_points"]]]
    ex = Exercise(c, ex4_render, rows, title='¿Qué genes, qué muestras?')
    ex.show()
    return ex


# ══════════════════════════════════════════════════════════════════════════════════════════════════
#  Exercise 5: raw counts vs counts per million (CPM)
# ══════════════════════════════════════════════════════════════════════════════════════════════════
VIEWS5 = [('Datos reales: mismos donantes, PBS vs una citocina', "pair"), ('Datos reales: todas las muestras de un tipo celular', "all"), ('Ejemplo de juguete (simulado)', "toy")]


def _geo_ratio(a, b):
    """Typical (geometric-mean) ratio b / a over donors, with +1 so that zeros do not break it."""
    return float(np.exp(np.mean(np.log((np.asarray(b, float) + 1) / (np.asarray(a, float) + 1)))))


def _ratio_text(r, what):
    if r >= 1.15:
        return f'<b>{r:.1f} veces más</b>'
    if r <= 1 / 1.15:
        return f'<b>{1 / r:.1f} veces menos</b>'
    return '<b>prácticamente igual</b>'


def _paired(ax, a, b, labels, title, colors=(PBS_COLOR, ORANGE), log=True):
    for x, y in zip(a, b):
        ax.plot([0, 1], [x, y], color=UP_COLOR if y > x else DOWN_COLOR, lw=1.1, alpha=0.65, zorder=1)
    ax.scatter(np.zeros(len(a)), a, s=26, color=colors[0], zorder=2)
    ax.scatter(np.ones(len(b)), b, s=26, color=colors[1], zorder=2)
    ax.plot([0, 1], [np.exp(np.mean(np.log(np.maximum(a, 0.5)))), np.exp(np.mean(np.log(np.maximum(b, 0.5))))], color="k", lw=3, marker="D", ms=7, zorder=3)
    ax.set_xticks([0, 1]); ax.set_xticklabels(labels); ax.set_xlim(-0.4, 1.4)
    if log:
        ax.set_yscale("log")
    ax.set_title(title)


def ex5_pair(gene, cell_type, cytokine):
    names = lambda c: [f"{d}|{cell_type}|{c}" for d in DONORS]
    r0, r1 = adata.obs_names.get_indexer(names("PBS")), adata.obs_names.get_indexer(names(cytokine))
    j = gene_index(gene)
    lib0, lib1 = adata.obs["library_size"].values[r0].astype(float), adata.obs["library_size"].values[r1].astype(float)
    raw0, raw1 = adata.X[r0, j].astype(float), adata.X[r1, j].astype(float)
    cpm0, cpm1 = adata.layers["cpm"][r0, j].astype(float), adata.layers["cpm"][r1, j].astype(float)
    cl = cyt_label(cytokine)
    display(HTML(
        f'''<div class=\"cx-info\">En <b>{CT_LABEL[cell_type]}</b>, con {cl} el tamaño de librería es {_ratio_text(_geo_ratio(lib0, lib1), "")} que con PBS. Los conteos crudos de <i>{gene}</i> son {_ratio_text(_geo_ratio(raw0, raw1), "")}. Tras el CPM, <i>{gene}</i> es {_ratio_text(_geo_ratio(cpm0, cpm1), "")} ({cl} / PBS, donante típico).</div>'''))
    fig, axes = plt.subplots(1, 3, figsize=(11.5, 3.9))
    lab = ["PBS", cl]
    _paired(axes[0], lib0, lib1, lab, '① Tamaño de librería de la muestra'); axes[0].set_ylabel('conteos totales de la muestra')
    _paired(axes[1], raw0, raw1, lab, f"② Conteos CRUDOS de {gene}"); axes[1].set_ylabel('conteos crudos')
    _paired(axes[2], cpm0, cpm1, lab, f"③ CPM de {gene}  (normalizado)"); axes[2].set_ylabel('conteos por millón (CPM)')
    fig.suptitle(f"{CT_LABEL[cell_type]}: una línea por donante (rojo = más alto con {cl}, azul = más bajo).  Negro = donante típico", fontsize=10, y=1.0)
    fig.tight_layout()
    show_fig(fig)


def ex5_all(gene, cell_type):
    m = adata.obs["cell_type"].values == cell_type
    j = gene_index(gene)
    lib = adata.obs["library_size"].values[m].astype(float)
    ctrl = (adata.obs["treatment"].values[m] == "control")
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.1), sharex=True)
    for ax, y, name in zip(axes, (adata.X[m, j].astype(float), adata.layers["cpm"][m, j].astype(float)), ('conteos crudos', "CPM")):
        ax.scatter(lib[~ctrl], y[~ctrl] + 1, s=8, alpha=0.45, color=CT_COLORS[cell_type], linewidths=0, label='muestras tratadas con citocina')
        ax.scatter(lib[ctrl], y[ctrl] + 1, s=36, facecolors="none", edgecolors="k", linewidths=1.1, label='controles PBS')
        ax.set_xscale("log"); ax.set_yscale("log")
        ok = y >= 0
        r = np.corrcoef(np.log10(lib), np.log10(y + 1))[0, 1] if np.std(y) > 0 else float("nan")
        if np.isfinite(r):
            slope, icpt = np.polyfit(np.log10(lib), np.log10(y + 1), 1)
            xs = np.array([lib.min(), lib.max()]); ax.plot(xs, 10 ** (icpt + slope * np.log10(xs)), color=UP_COLOR, lw=1.8)
        ax.set_xlabel('tamaño de librería de la muestra (conteos totales)'); ax.set_ylabel(f"{name} + 1")
        ax.set_title(f"{gene}: {name}\ncorrelación con el tamaño de librería: r = {r:+.2f}")
    axes[0].legend(loc="upper left", markerscale=1.6)
    fig.suptitle(f"{CT_LABEL[cell_type]}: {m.sum():,} muestras (cada punto es una muestra pseudobulk)", fontsize=10, y=1.0)
    fig.tight_layout()
    show_fig(fig)


def ex5_toy(depth, fold):
    names = ['gen A', 'gen B\n(gen de interferón)', 'gen C', 'gen D']
    base = np.array([1000.0, 50.0, 2500.0, 1450.0])                    # sample 1: PBS, deep (5,000 counts in total)
    p2 = base / base.sum(); p2[1] *= fold; p2 /= p2.sum()               # sample 2: gene B truly goes `fold` times up, the others do not change
    n2 = base.sum() * depth
    raw = np.vstack([base, np.round(p2 * n2)])
    cpm = raw / raw.sum(axis=1, keepdims=True) * 1e6
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.9))
    x, w = np.arange(4), 0.38
    for ax, M, title, yl in zip(axes, (raw, cpm), ('Conteos CRUDOS', 'CONTEOS POR MILLÓN (CPM)'), ("counts", "CPM")):
        ax.bar(x - w / 2, M[0], w, color=PBS_COLOR, label=f"muestra 1: PBS ({raw[0].sum():,.0f} conteos en total)")
        ax.bar(x + w / 2, M[1], w, color=ORANGE, label=f"muestra 2: con citocina ({raw[1].sum():,.0f} conteos en total)")
        for xi, (a, b) in enumerate(zip(M[0], M[1])):
            ax.text(xi, max(a, b) * 1.03, f"×{b / a:.2f}", ha="center", fontsize=9, fontweight="bold", color=UP_COLOR if b / a > 1.15 else (DOWN_COLOR if b / a < 0.87 else "#333"))
        ax.set_xticks(x); ax.set_xticklabels(names, fontsize=9); ax.set_ylabel(yl); ax.set_title(title)
        ax.set_ylim(0, M.max() * 1.18)
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=2, bbox_to_anchor=(0.5, -0.04), fontsize=9)
    fig.suptitle(f"Solo el gen B cambia de verdad (×{fold:g}). Números sobre las barras: muestra 2 / muestra 1", fontsize=10, y=1.0)
    fig.tight_layout(rect=(0, 0.05, 1, 1))
    show_fig(fig)
    display(HTML(f'''<div class=\"cx-info\">La muestra 2 se secuenció con <b>{depth:.0%}</b> de la profundidad de la muestra 1. En conteos crudos <b>todos los genes parecen {'más bajos' if depth < 0.95 else 'similares'}</b>, incluso el gen B (que de verdad sube ×{fold:g}). Tras dividir por el tamaño de librería (CPM), los genes A, C y D quedan casi igual y el gen B muestra su aumento real.</div>'''))


def ex5_render(view, gene, cell_type, cytokine, depth, fold):
    if view == "toy":
        return ex5_toy(depth, fold)
    g = one_gene(gene, None)
    if g is None:
        msg('Escribe un <b>símbolo génico</b> válido (por ejemplo <i>ACTB</i>, <i>STAT1</i>, <i>IDO1</i>).', "info"); return
    if view == "pair":
        if cytokine == "PBS":
            msg('Elige una citocina distinta de PBS.', "info"); return
        ex5_pair(g, cell_type, cytokine)
    else:
        ex5_all(g, cell_type)


def exercise_normalization():
    cyts = [c for c in adata.obs["cytokine"].cat.categories if c != "PBS"]
    c = dict(view=widgets.ToggleButtons(options=VIEWS5, value="pair", style={"button_width": "300px"}, layout=_lay("930px")),
             gene=gene_combo("ACTB", "170px"), cell_type=dropdown([(CT_LABEL[x], x) for x in CELL_TYPES], "CD14_Mono", "190px"),
             cytokine=dropdown([(cyt_label(x), x) for x in cyts], "IFN-gamma", "190px"),
             depth=float_slider(0.1, 1.0, 0.2, 0.05, "280px", ".0%"), fold=float_slider(1, 8, 4, 0.5, "280px", ".1f"))
    f_gene, f_ct, f_cyt = field('Gen', c["gene"], 'escribe un símbolo, p. ej. ACTB, STAT1, IDO1'), field('Tipo celular', c["cell_type"]), field('Comparar PBS con…', c["cytokine"])
    f_depth = field('Profundidad de secuenciación de la muestra 2 (relativa a la 1)', c["depth"])
    f_fold = field('Cambio real (fold change) del gen B', c["fold"])

    def _toggle(*_):
        real = c["view"].value != "toy"
        for f in (f_gene, f_ct):
            f.layout.display = None if real else "none"
        f_cyt.layout.display = None if c["view"].value == "pair" else "none"
        for f in (f_depth, f_fold):
            f.layout.display = "none" if real else None
    c["view"].observe(_toggle, "value"); _toggle()
    ex = Exercise(c, ex5_render, [[c["view"]], [f_gene, f_ct, f_cyt, f_depth, f_fold]], title='¿Conteos crudos o CPM?')
    ex.show()
    return ex


# ══════════════════════════════════════════════════════════════════════════════════════════════════
#  Exercise 6: logarithms: scale, spread and the zeros
# ══════════════════════════════════════════════════════════════════════════════════════════════════
BASES = [('2  (log₂, el que usamos)', "2"), ('e  (logaritmo natural)', "e"), ('10  (log₁₀)', "10")]
BASE_VALUE = {"2": 2.0, "e": math.e, "10": 10.0}
PSEUDO_OPTIONS = [0, 0.1, 0.5, 1, 2, 5, 10]
_MV_CACHE = {}


def log_transform(cpm, base, pseudo):
    """log_base(cpm + pseudocount). With pseudocount 0 the zeros become NaN (log of 0 does not exist)."""
    with np.errstate(divide="ignore", invalid="ignore"):
        out = np.log(cpm.astype(np.float64) + pseudo) / math.log(BASE_VALUE[base])
    out[~np.isfinite(out)] = np.nan
    return out


def mean_var(M):
    """Mean and variance of every column of M ignoring NaN (columns with fewer than 20 valid values → NaN)."""
    valid = np.isfinite(M).sum(axis=0)
    mean, var = np.nanmean(M, axis=0), np.nanvar(M, axis=0, ddof=1)
    bad = valid < 20
    mean[bad] = np.nan; var[bad] = np.nan
    return mean, var


def top20_share(var):
    v = var[np.isfinite(var)]
    return np.sort(v)[-20:].sum() / v.sum()


def ex6_render(gene, cell_type, base, pseudo):
    g = one_gene(gene, None)
    if g is None:
        msg('Escribe un <b>símbolo génico</b> válido (por ejemplo <i>CXCL10</i>, <i>IDO1</i>, <i>ACTB</i>).', "info"); return
    m = adata.obs["cell_type"].values == cell_type
    cpm_all = adata.layers["cpm"][m][:, EXPRESSED]
    key = ("cpm", cell_type)
    if key not in _MV_CACHE:                                                  # the CPM statistics never change: compute them once
        _MV_CACHE[key] = mean_var(cpm_all.astype(np.float64))
    mean_c, var_c = _MV_CACHE[key]
    L = log_transform(cpm_all, base, pseudo)
    mean_l, var_l = mean_var(L)
    j = gene_index(g)
    x = adata.layers["cpm"][m, j].astype(float)
    lx = log_transform(x, base, pseudo)
    n_zero = int((x == 0).sum())
    bname = {"2": "log₂", "e": "ln", "10": "log₁₀"}[base]
    fig, axes = plt.subplots(2, 2, figsize=(10.6, 7.6))
    col = CT_COLORS[cell_type]
    # (a) the gene, in CPM
    ax = axes[0, 0]
    ax.hist(x, bins=40, color=col, alpha=0.85)
    ax.axvline(np.mean(x), color="k", ls="--", lw=1.5); ax.axvline(np.median(x), color=UP_COLOR, ls=":", lw=2)
    ax.set_xlabel("CPM"); ax.set_ylabel('número de muestras')
    sk = stats.skew(x)
    ax.set_title(f"① {g} en CPM: asimétrico (asimetría {sk:.1f})")
    ax.legend(handles=[Line2D([], [], color="k", ls="--", label=f"media {np.mean(x):,.0f}"), Line2D([], [], color=UP_COLOR, ls=":", lw=2, label=f"mediana {np.median(x):,.0f}")], loc="upper right")
    # (b) the same gene after the logarithm
    ax = axes[0, 1]
    ok = np.isfinite(lx)
    ax.hist(lx[ok], bins=40, color=col, alpha=0.85)
    ax.set_xlabel(f"{bname}(CPM + {pseudo:g})" if pseudo else f"{bname}(CPM)"); ax.set_ylabel('número de muestras')
    sk2 = stats.skew(lx[ok]) if ok.sum() > 2 else float("nan")
    ax.set_title(f"② {g} tras el logaritmo (asimetría {sk2:.1f})")
    ax.text(0.02, 0.95, f"varianza de los valores:\n{np.var(x, ddof=1):,.0f}  →  {np.var(lx[ok], ddof=1):.2f}", transform=ax.transAxes, ha="left", va="top", fontsize=9.5)
    # (c) all genes: variance vs mean, CPM
    ax = axes[1, 0]
    ok = (mean_c > 0.01) & (var_c > 0)
    ax.scatter(mean_c[ok], var_c[ok], s=3, alpha=0.25, color=BLUE, linewidths=0, rasterized=True)
    xx = np.array([0.01, mean_c[ok].max()]); ax.plot(xx, xx ** 2 * 0.3, color="k", ls="--", lw=1.1, label='varianza ∝ media²')
    ax.set_xscale("log"); ax.set_yscale("log"); ax.legend(loc="upper left")
    ax.set_xlabel('CPM medio del gen'); ax.set_ylabel('varianza entre muestras')
    ax.set_title(f"③ CPM: la varianza crece con la media\n20 genes acumulan {top20_share(var_c):.0%} de la varianza total")
    # (d) all genes: variance vs mean, log
    ax = axes[1, 1]
    ok = np.isfinite(mean_l) & np.isfinite(var_l) & (var_l > 0)
    ax.scatter(mean_l[ok], var_l[ok], s=3, alpha=0.25, color=BLUE, linewidths=0, rasterized=True)
    ax.set_xlabel(f"media de {bname}(CPM{' + ' + format(pseudo, 'g') if pseudo else ''})"); ax.set_ylabel('varianza entre muestras')
    ax.set_title(f"④ Log: la varianza no depende de la expresión\n20 genes acumulan {top20_share(var_l):.1%} de la varianza total")
    fig.suptitle(f"{CT_LABEL[cell_type]} · {m.sum():,} muestras · {cpm_all.shape[1]:,} genes expresados", fontsize=10.5, y=1.0)
    fig.tight_layout()
    show_fig(fig)
    notes = []
    if pseudo == 0:
        notes.append(f"Sin <b>pseudoconteo</b>, log(0) no existe: " + (f"los {n_zero} valores cero de {g} ({n_zero / len(x):.0%}) no se pueden representar, y " if n_zero else f"{g} no tiene ceros aquí, pero ") + 'los genes con muchos ceros se eliminan del panel ④.')
    else:
        notes.append(f"El pseudoconteo ({pseudo:g}) se suma <i>antes</i> del logaritmo para que los ceros pasen a ser log({pseudo:g}) en lugar de −∞. Pseudoconteos mayores aplastan aún más los valores bajos.")
    if base != "2":
        notes.append(f"Cambiar la base solo <b>reescala</b> el eje (por una constante: log₂(x) = {math.log2(BASE_VALUE[base]):.2f} × {bname}(x)): la forma del histograma no cambia, las varianzas se multiplican por {(1 / math.log2(BASE_VALUE[base])) ** 2:.2f}).")
    display(HTML('<div class="cx-info">' + "<br>".join(notes) + "</div>"))


def exercise_logtransform():
    c = dict(gene=gene_combo("CXCL10", "170px"), cell_type=dropdown([(CT_LABEL[x], x) for x in CELL_TYPES], "CD14_Mono", "190px"),
             base=dropdown(BASES, "2", "220px"),
             pseudo=widgets.SelectionSlider(options=PSEUDO_OPTIONS, value=1, continuous_update=False, layout=_lay("330px"), style=STYLE))
    rows = [[field('Gen (paneles superiores)', c["gene"], 'p. ej. CXCL10, IDO1, ACTB'), field('Tipo celular', c["cell_type"]), field('Base del logaritmo', c["base"]),
             field('Pseudoconteo sumado antes del log', c["pseudo"], '0 = sin pseudoconteo: log(0) no está definido')]]
    ex = Exercise(c, ex6_render, rows, title='Transforma los valores de CPM')
    ex.show()
    return ex


# ══════════════════════════════════════════════════════════════════════════════════════════════════
#  Exercise 7: heatmaps of raw counts vs normalised values
# ══════════════════════════════════════════════════════════════════════════════════════════════════
HEAT_VIEWS = [('Conteos crudos', "raw"), ('Normalizado: log₂(CPM + 1)', "norm"), ('Ambos, lado a lado', "both")]
HEAT_COLS = [('cada donante (12 columnas por condición)', "donors"), ('la media de los 12 donantes', "mean")]
HEAT_DEFAULT_GROUPS = ['Marcadores de monocitos', 'Marcadores de células T', 'Marcadores de células T CD8', 'Marcadores de células NK', 'Respuesta a IFN-γ / STAT1',
                       'Respuesta a IL-4 / STAT6', 'Respuesta a IL-1β / NF-κB', 'Respuesta a IL-2 / STAT5']
HEAT_DEFAULT_CYTOKINES = ["PBS", "IFN-gamma", "IL-4", "IL-1-beta", "IL-2"]
MAX_HEAT_CONDITIONS, MAX_HEAT_GENES = 25, 70


def zscore_rows(M):
    """z-score every ROW (gene) across its columns (samples): (x − mean) / standard deviation."""
    sd = M.std(axis=1, ddof=1, keepdims=True)
    z = (M - M.mean(axis=1, keepdims=True)) / np.where(sd > 0, sd, 1)
    return np.where(sd > 0, z, 0.0)


def heatmap_data(genes, cell_types, conditions, kind, cols_mode):
    """genes × columns matrix (+ library size per column). Columns are ordered: cell type > condition > donor."""
    cols = [f"{d}|{ct}|{cy}" for ct in cell_types for cy in conditions for d in DONORS]
    idx = adata.obs_names.get_indexer(cols)
    gi = [gene_index(g) for g in genes]
    src = adata.X if kind == "raw" else adata.layers["log2cpm"]
    M = np.asarray(src[np.ix_(idx, gi)], dtype=np.float64).T
    lib = np.log10(adata.obs["library_size"].values[idx].astype(float))
    if cols_mode == "mean":
        M = M.reshape(len(genes), -1, len(DONORS)).mean(axis=2)
        lib = lib.reshape(-1, len(DONORS)).mean(axis=1)
    return M, lib[None, :]


def draw_heatmaps(panels, genes, group_sizes, group_names, cell_types, conditions, cols_mode, lib, title):
    """`panels` = [(title, matrix), …]: same genes and columns, one panel each, with a cell-type bar and a library-size bar on top."""
    n_panel = len(panels)
    n_per_ct = len(conditions) * (len(DONORS) if cols_mode == "donors" else 1)
    n_cols = n_per_ct * len(cell_types)
    height = 0.22 * len(genes) + 2.3
    fig = plt.figure(figsize=(6.4 * n_panel + 1.6, height))
    gs = fig.add_gridspec(3, n_panel + 1, height_ratios=[0.7, 0.7, max(len(genes), 4)], width_ratios=[1] * n_panel + [0.025],
                          hspace=0.06, wspace=0.05, top=1 - 0.55 / height, bottom=0.95 / height + 0.04, left=0.12 if n_panel == 1 else 0.075, right=0.94)
    im = None
    for c, (name, M) in enumerate(panels):
        ax_ct, ax_lib, ax = (fig.add_subplot(gs[r, c]) for r in range(3))
        for k, ct in enumerate(cell_types):
            ax_ct.add_patch(plt.Rectangle((k * n_per_ct - 0.5, 0), n_per_ct, 1, color=CT_COLORS[ct], lw=0))
            ax_ct.text(k * n_per_ct + n_per_ct / 2 - 0.5, 0.5, CT_SHORT[ct], ha="center", va="center", fontsize=9, fontweight="bold", color="black" if ct == "NK_CD56low" else "white")
        ax_ct.set_xlim(-0.5, n_cols - 0.5); ax_ct.set_ylim(0, 1); ax_ct.axis("off"); ax_ct.set_title(name, fontsize=11, pad=6)
        ax_lib.imshow(lib, aspect="auto", cmap="Greys", interpolation="nearest")
        ax_lib.set_xticks([]); ax_lib.set_yticks([0] if c == 0 else [])
        if c == 0:
            ax_lib.set_yticklabels(['tamaño de librería'], fontsize=7.5)
        for sp in ax_lib.spines.values():
            sp.set_visible(False)
        im = ax.imshow(zscore_rows(M), aspect="auto", cmap=DIVERGING, vmin=-2.5, vmax=2.5, interpolation="nearest")
        ax.set_yticks(range(len(genes))); ax.set_yticklabels(genes if c == 0 else [], fontsize=8, fontstyle="italic")
        if cols_mode == "donors":
            ax.set_xticks([(k + 0.5) * len(DONORS) - 0.5 for k in range(len(cell_types) * len(conditions))])
            ax.set_xticklabels([cyt_label(cy) for _ in cell_types for cy in conditions], rotation=90, fontsize=7)
        else:
            ax.set_xticks(range(n_cols)); ax.set_xticklabels([cyt_label(cy) for _ in cell_types for cy in conditions], rotation=90, fontsize=7.5)
        for b in np.cumsum(group_sizes)[:-1] - 0.5:
            ax.axhline(b, color="white", lw=2)
        for k in range(1, len(cell_types)):
            ax.axvline(k * n_per_ct - 0.5, color="white", lw=2)
        for sp in ax.spines.values():
            sp.set_visible(False)
        if c == 0 and group_names:
            starts = np.concatenate([[0], np.cumsum(group_sizes)[:-1]])
            for gname, st, sz in zip(group_names, starts, group_sizes):
                ax.annotate(gname, xy=(0, st + sz / 2 - 0.5), xycoords=("axes fraction", "data"), xytext=(-52, 0), textcoords="offset points",
                            ha="right", va="center", fontsize=7.5, color="#555", annotation_clip=False)
    fig.colorbar(im, cax=fig.add_subplot(gs[2, n_panel]), label='z-score del gen')
    fig.suptitle(title, y=1 - 0.06 / height, fontsize=10.5, fontweight="bold")
    show_fig(fig)


def ex7_render(view, groups, extra_text, cell_types, cytokines, cols_mode):
    conditions = [c for c in adata.obs["cytokine"].cat.categories if c in set(cytokines)]
    cell_types = [c for c in CELL_TYPES if c in set(cell_types)]
    genes, group_sizes, group_names = [], [], []
    for gname in groups:
        gg = [g for g in GENESETS.get(gname, []) if g not in genes]
        if gg:
            genes += gg; group_sizes.append(len(gg)); group_names.append(gname)
    extra, unknown = parse_genes(extra_text, 40)
    unknown_genes_message(unknown)
    extra = [g for g in extra if g not in genes]
    if extra:
        genes += extra; group_sizes.append(len(extra)); group_names.append('tus genes')
    if not genes:
        msg('Selecciona al menos un <b>grupo de genes</b> o escribe algunos símbolos génicos.', "info"); return
    if not conditions or not cell_types:
        msg('Selecciona al menos un <b>tipo celular</b> y una <b>citocina / condición</b>.', "info"); return
    if len(conditions) > MAX_HEAT_CONDITIONS:
        msg(f"Demasiadas condiciones para un mapa de calor legible: se muestran las primeras {MAX_HEAT_CONDITIONS}.", "warn"); conditions = conditions[:MAX_HEAT_CONDITIONS]
    if len(genes) > MAX_HEAT_GENES:
        msg(f"Demasiados genes para un mapa de calor legible: se muestran los primeros {MAX_HEAT_GENES}.", "warn")
        genes = genes[:MAX_HEAT_GENES]; group_sizes = list(np.minimum(group_sizes, np.maximum(0, MAX_HEAT_GENES - np.concatenate([[0], np.cumsum(group_sizes)[:-1]]))).astype(int))
        group_names = [n for n, s in zip(group_names, group_sizes) if s > 0]; group_sizes = [s for s in group_sizes if s > 0]
    panels, lib = [], None
    for kind, name in (("raw", 'ANTES: conteos crudos'), ("norm", 'DESPUÉS: log₂(CPM + 1)')):
        if view == "both" or view == kind:
            M, lib = heatmap_data(genes, cell_types, conditions, kind, cols_mode)
            panels.append((name, M))
    n_cols = len(cell_types) * len(conditions) * (len(DONORS) if cols_mode == "donors" else 1)
    draw_heatmaps(panels, genes, group_sizes, group_names, cell_types, conditions, cols_mode, lib,
                  f"{len(genes)} genes × {n_cols} columnas ({len(cell_types)} tipos celulares × {len(conditions)} condiciones" + (' × 12 donantes)' if cols_mode == "donors" else ', media de 12 donantes)'))


def exercise_heatmaps():
    cyts = list(adata.obs["cytokine"].cat.categories)
    c = dict(view=widgets.ToggleButtons(options=HEAT_VIEWS, value="both", style={"button_width": "250px"}),
             groups=multi_select(list(GENESETS), pick_default(list(GENESETS), HEAT_DEFAULT_GROUPS), rows=10, width="240px"),
             extra_text=None, cell_types=multi_select([(CT_LABEL[x], x) for x in CELL_TYPES], CELL_TYPES, rows=4, width="170px"),
             cytokines=multi_select(cyts, pick_default(cyts, HEAT_DEFAULT_CYTOKINES), rows=10, width="170px"),
             cols_mode=dropdown(HEAT_COLS, "donors", "290px"))
    ttext, twidget = gene_picker("", "260px", 'p. ej. TNF, IL1B', with_sets=False, label='Genes extra (opcional)')
    c["extra_text"] = ttext
    rows = [[c["view"]],
            [widgets.VBox([field('Grupos de genes', c["groups"], 'Ctrl/⌘ + clic: varios'), selection_summary(c["groups"], max_items=3)]), widgets.VBox([twidget]),
             widgets.VBox([field('Tipos celulares', c["cell_types"]), field('Las columnas muestran', c["cols_mode"])]),
             widgets.VBox([field('Citocinas / condiciones', c["cytokines"], 'PBS = control'), selection_summary(c["cytokines"], cyt_label)])]]
    ex = Exercise(c, ex7_render, rows, title='¿Qué quieres ver?')
    ex.show()
    return ex


# ══════════════════════════════════════════════════════════════════════════════════════════════════
#  Exercise 8: the geometry of PCA with two genes (rotation and projection)
# ══════════════════════════════════════════════════════════════════════════════════════════════════
PAIRS8 = {"corr": ("ISG15", "MX1"), "uncorr": ("IDO1", "IL1B")}
PAIR_CHOICES = [('Genes correlacionados: ISG15 y MX1 (dos genes de respuesta a interferón)', "corr"), ('Genes no correlacionados: IDO1 e IL1B (dos rutas distintas)', "uncorr")]


def pair_data(pair):
    """Centred log2(CPM + 1) of the two genes in the monocyte samples + everything PCA would compute from them."""
    a, b = PAIRS8[pair]
    m = adata.obs["cell_type"].values == "CD14_Mono"
    P = np.c_[adata.layers["log2cpm"][m, gene_index(a)], adata.layers["log2cpm"][m, gene_index(b)]].astype(np.float64)
    Pc = P - P.mean(axis=0)
    C = np.cov(Pc.T)
    w, V = np.linalg.eigh(C)                                                  # PCA of two variables: eigen-decomposition of the covariance
    w, V = w[::-1], V[:, ::-1]
    if V[np.argmax(np.abs(V[:, 0])), 0] < 0:
        V[:, 0] = -V[:, 0]
    theta1 = np.degrees(np.arctan2(V[1, 0], V[0, 0])) % 180
    is_ifn = adata.obs.loc[adata.obs["cell_type"] == "CD14_Mono", "cytokine"].astype(str).str.startswith("IFN-").values
    return dict(genes=(a, b), Pc=Pc, C=C, eigvals=w, V=V, theta1=theta1, r=np.corrcoef(Pc.T)[0, 1], is_ifn=is_ifn)


def ex8_render(pair, angle, show_proj, show_pc):
    d = pair_data(pair)
    a, b = d["genes"]
    Pc, C = d["Pc"], d["C"]
    th = np.radians(angle)
    u = np.array([np.cos(th), np.sin(th)])                                   # direction of our axis
    v = np.array([-np.sin(th), np.cos(th)])                                  # the perpendicular one
    s_u, s_v = Pc @ u, Pc @ v                                                # coordinates in the rotated system
    total = np.trace(C)
    captured = u @ C @ u
    best = d["eigvals"][0]
    fig, axes = plt.subplots(1, 3, figsize=(13, 4.5), gridspec_kw={"width_ratios": [1.15, 1, 1.15]})
    # ① the data with the axis and the projections
    ax = axes[0]
    ax.scatter(Pc[:, 0], Pc[:, 1], s=7, alpha=0.4, color=NS_COLOR, linewidths=0, label='otras citocinas y PBS')
    ax.scatter(Pc[d["is_ifn"], 0], Pc[d["is_ifn"], 1], s=9, alpha=0.7, color=UP_COLOR, linewidths=0, label='interferones')
    L = 1.15 * np.abs(Pc).max()
    if show_proj:
        idx = np.random.default_rng(3).choice(len(Pc), 45, replace=False)
        for i in idx:
            p = (Pc[i] @ u) * u
            ax.plot([Pc[i, 0], p[0]], [Pc[i, 1], p[1]], color="k", lw=0.7, ls=":", alpha=0.8)
        ax.scatter(*(np.outer(Pc[idx] @ u, u)).T, s=12, color="k", zorder=5, label='proyección sobre el eje')
    ax.plot([-L * u[0], L * u[0]], [-L * u[1], L * u[1]], color=PURPLE, lw=2.6, zorder=4, label=f"tu eje ({angle}°)")
    if show_pc:
        for k, (col, nm) in enumerate(((DARK, "PC1"), ("#888", "PC2"))):
            e = d["V"][:, k] * 1.9 * np.sqrt(d["eigvals"][k])
            ax.annotate("", xy=e, xytext=-e, arrowprops=dict(arrowstyle="<|-|>", color=col, lw=1.6, ls="--"))
            ax.text(*(e * 1.08), nm, fontsize=10, fontweight="bold", color=col, ha="center", va="center")
    ax.axhline(0, color="#ccc", lw=0.6); ax.axvline(0, color="#ccc", lw=0.6)
    ax.set_xlim(-L, L); ax.set_ylim(-L, L); ax.set_aspect("equal")
    ax.set_xlabel(f"{a}  (log₂ CPM + 1, centrado)"); ax.set_ylabel(f"{b}  (log₂ CPM + 1, centrado)")
    ax.set_title(f"① {a} vs {b} en monocitos (r = {d['r']:.2f})")
    ax.legend(loc="lower right", fontsize=7.5, markerscale=1.4)
    # ② variance of the projections for every possible angle
    ax = axes[1]
    ang = np.arange(0, 181)
    var_ang = np.array([np.array([np.cos(np.radians(t)), np.sin(np.radians(t))]) @ C @ np.array([np.cos(np.radians(t)), np.sin(np.radians(t))]) for t in ang])
    ax.plot(ang, var_ang / total * 100, color=BLUE, lw=2.5)
    ax.scatter([angle], [captured / total * 100], s=90, color=PURPLE, zorder=5)
    ax.annotate(f"{captured / total:.0%}", xy=(angle, captured / total * 100), xytext=(0, 11), textcoords="offset points", ha="center", fontsize=10.5, fontweight="bold", color=PURPLE)
    if show_pc:
        ax.axvline(d["theta1"], color=DARK, ls="--", lw=1.4); ax.text(d["theta1"], 3, f" PC1: {d['theta1']:.0f}° ({best / total:.0%})", rotation=90, va="bottom", ha="right", fontsize=9)
    ax.set_ylim(0, 105); ax.set_xlim(0, 180); ax.set_xticks([0, 45, 90, 135, 180])
    ax.set_xlabel('ángulo del eje (grados)'); ax.set_ylabel('% de la varianza total que conserva la proyección')
    ax.set_title('② ¿Cuánta varianza conserva cada eje?')
    # ③ the data seen from the rotated axes
    ax = axes[2]
    ax.scatter(s_u, s_v, s=7, alpha=0.4, color=NS_COLOR, linewidths=0)
    ax.scatter(s_u[d["is_ifn"]], s_v[d["is_ifn"]], s=9, alpha=0.7, color=UP_COLOR, linewidths=0)
    ax.axhline(0, color=PURPLE, lw=1.2); ax.axvline(0, color="#ccc", lw=0.6)
    ax.set_xlim(-L, L); ax.set_ylim(-L, L); ax.set_aspect("equal")
    ax.set_xlabel(f"coordenada en tu eje  (varianza {np.var(s_u, ddof=1):.2f})"); ax.set_ylabel(f"coordenada en el eje perpendicular  (varianza {np.var(s_v, ddof=1):.2f})")
    ax.set_title(f"③ Las mismas muestras tras rotar {angle}°\ncorrelación entre las nuevas coordenadas: {np.corrcoef(s_u, s_v)[0, 1]:+.2f}")
    fig.tight_layout()
    show_fig(fig)
    if abs(((angle - d["theta1"] + 90) % 180) - 90) <= 2:
        msg(f'''🎯 Tu eje es (casi) la <b>PC1</b>: conserva la máxima varianza ({best / total:.0%}) y, en ③, las dos nuevas coordenadas ya no están correlacionadas: <b>el PCA es solo esta rotación</b>. Descartar el eje perpendicular (PC2) solo perdería {1 - best / total:.0%} de la varianza.''', "info")


def exercise_pca_geometry():
    c = dict(pair=dropdown(PAIR_CHOICES, "corr", "520px"), angle=widgets.IntSlider(min=0, max=180, value=0, step=1, continuous_update=False,
                                                                                     layout=_lay("430px"), style=STYLE, readout_format="d"),
             show_proj=checkbox('mostrar las proyecciones (líneas punteadas)', True, "270px"), show_pc=checkbox('mostrar PC1 y PC2', False, "200px"))
    snap = widgets.Button(description='🎯 ir a PC1', layout=_lay("140px"))
    snap.on_click(lambda _b: setattr(c["angle"], "value", int(round(pair_data(c["pair"].value)["theta1"])) % 181))
    rows = [[field('Elige el par de genes', c["pair"])],
            [field('Rota tu eje: ángulo (grados)', c["angle"], '0° = el eje x (primer gen), 90° = el eje y (segundo gen)'),
             widgets.VBox([widgets.HTML('<div class="cx-label">&nbsp;</div>'), snap]), widgets.VBox([widgets.HTML('<div class="cx-label">&nbsp;</div>'), c["show_proj"]]),
             widgets.VBox([widgets.HTML('<div class="cx-label">&nbsp;</div>'), c["show_pc"]])]]
    ex = Exercise(c, ex8_render, rows, title='Rota un eje y observa la proyección')
    ex.show()
    return ex


# ══════════════════════════════════════════════════════════════════════════════════════════════════
#  Exercises 9 and 10: PCA scatter plots (all samples, and one cell type at a time)
# ══════════════════════════════════════════════════════════════════════════════════════════════════
def pca_scatter(ax, P, color_by, pcs=(0, 1), cytokines=(), gene=None, s=14):
    """Scatter of two PCs. color_by: cell_type | donor | treatment | cytokine | n_cells | library_size | gene."""
    sc, (a, b), obs = P["scores"], pcs, P["obs"]
    kw = dict(linewidths=0.3, edgecolors="white")
    legend = True
    if color_by == "cell_type":
        for ct in CELL_TYPES:
            m = (obs["cell_type"] == ct).values
            if m.any():
                ax.scatter(sc[m, a], sc[m, b], s=s, alpha=0.8, color=CT_COLORS[ct], marker=CT_MARKERS[ct], label=CT_LABEL[ct], **kw)
    elif color_by == "donor":
        for i, dn in enumerate(DONORS):
            m = (obs["donor"] == dn).values
            ax.scatter(sc[m, a], sc[m, b], s=s, alpha=0.85, color=DONOR_HUES[i % 6], marker="o" if i < 6 else "^", label=dn, **kw)
    elif color_by == "treatment":
        m = (obs["treatment"] == "treated").values
        ax.scatter(sc[m, a], sc[m, b], s=s * 0.8, color=NS_COLOR, alpha=0.6, linewidths=0, label=f"tratadas con una citocina ({m.sum():,})", zorder=1)
        ax.scatter(sc[~m, a], sc[~m, b], s=s * 2.6, color="black", label=f"control, PBS ({(~m).sum():,})", zorder=2, **kw)
    elif color_by == "cytokine":
        order = {c: i for i, c in enumerate(adata.obs["cytokine"].cat.categories)}
        picked = sorted(cytokines, key=lambda c: order.get(c, 999))[:len(CYT_HUES)]          # same colour for the same cytokine, whatever the click order
        m = ~obs["cytokine"].isin(["PBS"] + picked).values
        ax.scatter(sc[m, a], sc[m, b], s=s * 0.6, color=NS_COLOR, alpha=0.5, linewidths=0, label='todas las demás citocinas', zorder=1)
        for i, cy in enumerate([c for c in picked if c != "PBS"]):
            m = (obs["cytokine"] == cy).values
            ax.scatter(sc[m, a], sc[m, b], s=s * 1.5, color=CYT_HUES[i], marker=CYT_MARKERS[i], label=cyt_label(cy), zorder=2, **kw)
        m = (obs["cytokine"] == "PBS").values
        ax.scatter(sc[m, a], sc[m, b], s=s * 2.6, color="black", label='PBS (control)', zorder=3, **kw)
    else:                                                                       # a number: n_cells, library_size or a gene
        legend = False
        if color_by == "gene":
            values, label = P_gene_values(P, gene), f"{gene}\nlog₂(CPM + 1)"
        elif color_by == "library_size":
            values, label = np.log10(obs["library_size"].values.astype(float)), 'log₁₀ tamaño de librería'
        else:
            values, label = np.log10(obs["psbulk_n_cells"].values.astype(float)), 'log₁₀ número de células'
        order = np.argsort(values)                                              # highest values on top
        pts = ax.scatter(sc[order, a], sc[order, b], c=values[order], cmap=SEQ_POINTS, s=s * 1.5, **kw)
        plt.colorbar(pts, ax=ax, shrink=0.8, pad=0.01, label=label)
    if legend:
        n_leg = len(ax.get_legend_handles_labels()[0])
        ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), fontsize=8.5, markerscale=1.5, handletextpad=0.3, columnspacing=1.2, ncol=min(n_leg, 6))
    ax.set_xlabel(f"PC{a + 1}  ({P['var'][a]:.1f} % de la varianza)"); ax.set_ylabel(f"PC{b + 1}  ({P['var'][b]:.1f} %)")
    ax.axhline(0, color="#ddd", lw=0.6, zorder=0); ax.axvline(0, color="#ddd", lw=0.6, zorder=0)


def P_gene_values(P, gene):
    return adata.layers["log2cpm"][P["mask"], gene_index(gene)].astype(float)


def plot_scree(ax, P, pcs, n=10):
    var = P["var"][:n]
    ax.bar(range(1, len(var) + 1), var, color=[ORANGE if i in pcs else "#9bbbe6" for i in range(len(var))])
    for i, v in enumerate(var[:6]):
        ax.text(i + 1, v + 0.8, f"{v:.0f}%" if v >= 10 else f"{v:.1f}%", ha="center", fontsize=7.5)
    ax.set_xticks(range(1, len(var) + 1)); ax.set_xlabel('componente principal'); ax.set_ylabel('varianza explicada (%)'); ax.set_title('¿Cuánta varianza contiene cada PC?')


def plot_r2(ax, P, pcs, factors=None):
    """What does each of the chosen PCs capture? R² with every sample variable."""
    R2 = P["r2"]
    cols = [c for c in (factors or R2.columns) if c in R2.columns]
    y = np.arange(len(cols))
    w = 0.38
    for k, (pc, col) in enumerate(zip(pcs, (ORANGE, BLUE))):
        vals = [R2.loc[f"PC{pc + 1}", c] for c in cols]
        ax.barh(y + (k - 0.5) * w, vals, w, color=col, label=f"PC{pc + 1}")
        for yy, v in zip(y + (k - 0.5) * w, vals):
            if v > 0.03:
                ax.text(v + 0.01, yy, f"{v:.2f}", va="center", fontsize=7.5)
    ax.set_yticks(y); ax.set_yticklabels([disp(c) for c in cols], fontsize=8.5); ax.invert_yaxis(); ax.set_xlim(0, 1.12)
    ax.set_xlabel('R²  (fracción de la PC explicada por la variable)'); ax.set_title('¿Qué captura cada PC?'); ax.legend(loc="lower right")


COLOR_CHOICES = [(disp("Donor"), "donor"), ('Tipo celular', "cell_type"), ('Control vs tratado', "treatment"), ('Citocina (resalta algunas)', "cytokine"),
                 ('Número de células en la muestra', "n_cells"), ('Tamaño de librería', "library_size"), ('Expresión de un gen', "gene")]
PC_CHOICES = [(f"PC{i}", i - 1) for i in range(1, 11)]


def _pca_common_msgs(color_by, gene, cytokines):
    if color_by == "gene":
        g = one_gene(gene, None)
        if g is None:
            msg('Escribe un <b>símbolo génico</b> válido para colorear por su expresión (p. ej. <i>LYZ</i>, <i>CD3E</i>, <i>UTY</i>).', "info")
        return g
    if color_by == "cytokine" and len(cytokines) > len(CYT_HUES):
        msg(f"Solo se colorean las primeras {len(CYT_HUES)} citocinas (más colores no se distinguirían).", "warn")
    return None


def ex9_render(color_by, pc_x, pc_y, cytokines, gene):
    g = _pca_common_msgs(color_by, gene, cytokines)
    if color_by == "gene" and g is None:
        return
    if pc_x == pc_y:
        msg('Elige dos PC <b>distintas</b> para los dos ejes.', "info"); return
    P = PCA_ALL
    fig = plt.figure(figsize=(13, 5.8))
    gs = fig.add_gridspec(2, 2, width_ratios=[1.45, 1], height_ratios=[1, 1.15], hspace=0.55, wspace=0.28)
    ax = fig.add_subplot(gs[:, 0])
    pca_scatter(ax, P, color_by, (pc_x, pc_y), cytokines, g)
    ax.set_title(f"Las {adata.n_obs:,} muestras · PCA de los {N_HVG:,} genes más variables")
    plot_scree(fig.add_subplot(gs[0, 1]), P, (pc_x, pc_y))
    plot_r2(fig.add_subplot(gs[1, 1]), P, (pc_x, pc_y), ['tipo celular', "donor", "cytokine", 'control vs tratado', 'tamaño de librería', 'número de células'])
    show_fig(fig)


def exercise_pca_all():
    cyts = [c for c in adata.obs["cytokine"].cat.categories if c != "PBS"]
    c = dict(color_by=dropdown(COLOR_CHOICES, "donor", "280px"), pc_x=dropdown(PC_CHOICES, 0, "100px"), pc_y=dropdown(PC_CHOICES, 1, "100px"),
             cytokines=multi_select(cyts, pick_default(cyts, ["IFN-gamma", "IL-4", "TNF-alpha", "IL-10", "IL-6", "GM-CSF"]), rows=7, width="170px"),
             gene=gene_combo("LYZ", "160px"))
    f_cyt = widgets.VBox([field('Citocinas a resaltar', c["cytokines"], 'hasta 8; PBS siempre en negro'), selection_summary(c["cytokines"], cyt_label, 4)])
    f_gene = field('Gen', c["gene"], 'escribe un símbolo')

    def _toggle(*_):
        f_cyt.layout.display = None if c["color_by"].value == "cytokine" else "none"
        f_gene.layout.display = None if c["color_by"].value == "gene" else "none"
    c["color_by"].observe(_toggle, "value"); _toggle()
    ex = Exercise(c, ex9_render, [[field('Colorear las muestras por…', c["color_by"]), field('eje x', c["pc_x"]), field('eje y', c["pc_y"]), f_cyt, f_gene]], title='PCA de todos los datos')
    ex.show()
    return ex


# ── Exercise 10: one cell type at a time ──────────────────────────────────────────────────────────
COLOR_CHOICES_CT = [('Citocina (resalta algunas)', "cytokine"), (disp("Donor"), "donor"), ('Control vs tratado', "treatment"), ('Tamaño de librería', "library_size"), ('Expresión de un gen', "gene")]


def plot_loadings(ax, P, pc, n=8):
    """The genes with the largest weights (loadings) of one PC."""
    s = pd.Series(P["loadings"][:, pc], index=P["hvg_names"]).sort_values()
    top = pd.concat([s.head(n), s.tail(n)])
    ax.barh(top.index, top.values, color=[DOWN_COLOR if v < 0 else UP_COLOR for v in top.values])
    ax.axvline(0, color="k", lw=0.6); ax.tick_params(axis="y", labelsize=8)
    for lab in ax.get_yticklabels():
        lab.set_fontstyle("italic")
    ax.set_xlabel('loading (peso del gen)'); ax.set_title(f"Genes detrás de la PC{pc + 1}")


def ex10_render(cell_type, color_by, pc_x, pc_y, cytokines, gene, show_loadings):
    g = _pca_common_msgs(color_by, gene, cytokines)
    if color_by == "gene" and g is None:
        return
    if pc_x == pc_y:
        msg('Elige dos PC <b>distintas</b> para los dos ejes.', "info"); return
    P = PCA_CT[cell_type]
    if show_loadings:
        fig = plt.figure(figsize=(13.5, 5.4))
        gs = fig.add_gridspec(2, 2, width_ratios=[1.6, 1], hspace=0.55, wspace=0.55)
        ax = fig.add_subplot(gs[:, 0])
        plot_loadings(fig.add_subplot(gs[0, 1]), P, pc_x)
        plot_loadings(fig.add_subplot(gs[1, 1]), P, pc_y)
    else:
        fig, ax = plt.subplots(figsize=(8.6, 5.2))
    pca_scatter(ax, P, color_by, (pc_x, pc_y), cytokines, g, s=22)
    ax.set_title(f"{CT_LABEL[cell_type]} solo ({P['obs'].shape[0]:,} muestras): PCA de sus {N_HVG:,} genes más variables")
    show_fig(fig)
    R2 = P["r2"]
    rows = {f"PC{pc + 1}": {'varianza': f"{P['var'][pc]:.1f} %", 'R² donante': f"{R2.loc[f'PC{pc + 1}', 'donor']:.2f}", 'R² citocina': f"{R2.loc[f'PC{pc + 1}', 'cytokine']:.2f}",
                              'R² tamaño de librería': f"{R2.loc[f'PC{pc + 1}', 'tamaño de librería']:.2f}"} for pc in (pc_x, pc_y)}
    display(HTML('<div class="cx-label">¿Qué capturan estas dos PC en este tipo celular?</div>' + html_table(pd.DataFrame(rows).T, index_label="PC")))


def exercise_pca_celltype():
    cyts = [c for c in adata.obs["cytokine"].cat.categories if c != "PBS"]
    c = dict(cell_type=dropdown([(CT_LABEL[x], x) for x in CELL_TYPES], "CD14_Mono", "190px"), color_by=dropdown(COLOR_CHOICES_CT, "cytokine", "250px"),
             pc_x=dropdown(PC_CHOICES, 0, "100px"), pc_y=dropdown(PC_CHOICES, 3, "100px"),
             cytokines=multi_select(cyts, pick_default(cyts, ["IFN-beta", "IFN-gamma", "IL-4", "IL-13", "GM-CSF", "IL-10"]), rows=7, width="170px"),
             gene=gene_combo("UTY", "160px"), show_loadings=checkbox('mostrar los genes detrás de cada eje', True, "250px"))
    f_cyt = widgets.VBox([field('Citocinas a resaltar', c["cytokines"], 'hasta 8; PBS siempre en negro'), selection_summary(c["cytokines"], cyt_label, 4)])
    f_gene = field('Gen', c["gene"], 'escribe un símbolo')

    def _toggle(*_):
        f_cyt.layout.display = None if c["color_by"].value == "cytokine" else "none"
        f_gene.layout.display = None if c["color_by"].value == "gene" else "none"
    c["color_by"].observe(_toggle, "value"); _toggle()
    ex = Exercise(c, ex10_render, [[field('Tipo celular', c["cell_type"]), field('Colorear las muestras por…', c["color_by"]), field('eje x', c["pc_x"]), field('eje y', c["pc_y"]), f_cyt, f_gene],
                                   [c["show_loadings"]]], title='PCA dentro de un tipo celular')
    ex.show()
    return ex


# ══════════════════════════════════════════════════════════════════════════════════════════════════
#  Exercise 11: volcano plots of the pre-computed DESeq2 results (cytokine vs PBS, one cell type)
# ══════════════════════════════════════════════════════════════════════════════════════════════════
PADJ_OPTIONS = [0.001, 0.01, 0.05, 0.1, 0.2]
try:
    from adjustText import adjust_text
    HAS_ADJUSTTEXT = True
except Exception:
    HAS_ADJUSTTEXT = False


def de_summary(res, padj_thr, lfc_thr):
    """Add the up / down / ns call and the -log10 adjusted p-value to a DE table."""
    d = res.copy()
    d["direction"] = classify_de(d, padj_thr, lfc_thr)
    d["nlp"] = d["neglog10_padj"]
    d["score"] = d["nlp"] * d["log2fc"].abs()                                    # evidence × effect size: used to pick the genes to label
    return d


def mean_expression(gene_list, cell_type, cytokine):
    """Mean log2(CPM + 1) of each gene in the 12 PBS and the 12 cytokine samples of one cell type."""
    r0 = adata.obs_names.get_indexer([f"{d}|{cell_type}|PBS" for d in DONORS])
    r1 = adata.obs_names.get_indexer([f"{d}|{cell_type}|{cytokine}" for d in DONORS])
    gi = [gene_index(g) for g in gene_list]
    L = adata.layers["log2cpm"]
    return L[np.ix_(r0, gi)].mean(axis=0), L[np.ix_(r1, gi)].mean(axis=0)


def axis_caps(x, y):
    """A few genes have extreme values (e.g. log2FC = 12 when a gene is off in PBS): they are drawn on the edge instead of squeezing the plot."""
    xq, yq = np.quantile(np.abs(x), 0.999), np.quantile(y, 0.999)
    return min(np.abs(x).max(), max(2.0 * xq, 3.0)) * 1.04, min(y.max(), max(1.8 * yq, 10.0)) * 1.04


def draw_volcano(ax, d, padj_thr, lfc_thr, n_labels, highlight=(), title=""):
    t = d[d["tested"]].copy()
    xcap, ycap = axis_caps(t["log2fc"].values, t["nlp"].values)
    t["x"], t["y"] = t["log2fc"].clip(-xcap, xcap), t["nlp"].clip(upper=ycap)
    beyond = (t["x"] != t["log2fc"]) | (t["y"] != t["nlp"])
    for k, col, size, alpha in (("ns", NS_COLOR, 6, 0.55), ("down", DOWN_COLOR, 12, 0.8), ("up", UP_COLOR, 12, 0.8)):
        sub = t[(t["direction"] == k) & ~beyond]
        ax.scatter(sub["x"], sub["y"], s=size, color=col, alpha=alpha, linewidths=0, rasterized=True, zorder=2)
        sub = t[(t["direction"] == k) & beyond]
        ax.scatter(sub["x"], sub["y"], s=40, color=col, marker="^", linewidths=0.5, edgecolors="k", zorder=3)
    for x in (-lfc_thr, lfc_thr):
        ax.axvline(x, color="#999", lw=0.8, ls="--", zorder=1)
    ax.axhline(-np.log10(padj_thr), color="#999", lw=0.8, ls="--", zorder=1)
    ax.set_xlim(-xcap * 1.04, xcap * 1.04); ax.set_ylim(-ycap * 0.03, ycap * 1.1)
    x0, x1 = ax.get_xlim(); y0, y1 = ax.get_ylim()
    sig = t[t["direction"] != "ns"]
    hl = [g for g in highlight if g in t.index]
    texts = []
    if n_labels > 0 and len(sig):                                               # the strongest DEGs, half up and half down
        for k in ("up", "down"):
            top = sig[sig["direction"] == k].nlargest(max(1, n_labels // 2), "score")
            for g, r in top.iterrows():
                if g not in hl and not (r["nlp"] > y0 + 0.85 * (y1 - y0) and (r["log2fc"] < x0 + 0.25 * (x1 - x0) or r["log2fc"] > x1 - 0.25 * (x1 - x0))):
                    texts.append(ax.text(r["x"], r["y"], g, fontsize=7.5, fontstyle="italic", zorder=5))
    if hl:                                                                       # the genes chosen by the student
        r = t.loc[hl]
        ax.scatter(r["x"], r["y"], s=70, facecolors="none", edgecolors="k", linewidths=1.6, zorder=6)
        for g, rr in r.iterrows():
            texts.append(ax.text(rr["x"], rr["y"], g, fontsize=9.5, fontweight="bold", fontstyle="italic", zorder=7,
                                 bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="k", lw=0.6, alpha=0.9)))
    if HAS_ADJUSTTEXT and texts:
        try:
            adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle="-", color="#888", lw=0.5), expand=(1.2, 1.4), max_move=(30, 30))
        except Exception:
            pass
    n_up, n_down = int((t["direction"] == "up").sum()), int((t["direction"] == "down").sum())
    ax.annotate(f"↑ {n_up:,} al alza", xy=(0.98, 0.98), xycoords="axes fraction", ha="right", va="top", color=UP_COLOR, fontsize=11, fontweight="bold")
    ax.annotate(f"↓ {n_down:,} a la baja", xy=(0.02, 0.98), xycoords="axes fraction", ha="left", va="top", color=DOWN_COLOR, fontsize=11, fontweight="bold")
    foot = f"{len(t):,} genes analizados · {len(t) - n_up - n_down:,} no significativos" + (f"   ·   ▲ {int(beyond.sum())} fuera de los ejes (ver las tablas)" if beyond.any() else "")
    ax.text(0.5, 1.005, foot, transform=ax.transAxes, ha="center", va="bottom", color="#777", fontsize=8.5)
    ax.set_xlabel('log₂ fold change  (citocina / PBS)'); ax.set_ylabel('−log₁₀ p-valor ajustado')
    ax.set_title(title, pad=16)
    return n_up, n_down


def _status(r, padj_thr, lfc_thr):
    if not r["tested"]:
        return 'no analizado (fuera de la lista de genes inmunes, o muy pocos conteos)'
    return {"up": '<span class="cx-up">↑ sobre-expresado</span>', "down": '<span class="cx-down">↓ infra-expresado</span>', "ns": 'no significativo'}[r["direction"]]


def gene_stats_table(d, genes, cell_type, cytokine, padj_thr, lfc_thr):
    m0, m1 = mean_expression(genes, cell_type, cytokine)
    rows = []
    for g, a, b in zip(genes, m0, m1):
        r = d.loc[g]
        rows.append({"gene": g, 'log₂ fold change': r["log2fc"] if r["tested"] else np.nan, 'fold change': (2 ** r["log2fc"]) if r["tested"] else np.nan,
                     'p-valor ajustado': f"{r['padj']:.1e}" if r["tested"] else "—", 'media en PBS': a, f"media con {cyt_label(cytokine)}": b, 'veredicto': _status(r, padj_thr, lfc_thr)})
    df = pd.DataFrame(rows).set_index("gene")
    return html_table(df, float_fmt="{:,.2f}", left_cols=['veredicto'], index_label=disp("gene"))


def top_table(d, direction, n=10):
    t = d[(d["direction"] == direction)].nlargest(n, "score")
    df = pd.DataFrame({'log₂ fold change': t["log2fc"], 'fold change': 2.0 ** t["log2fc"], 'p-valor ajustado': t["padj"].map(lambda x: f"{x:.1e}")})
    return html_table(df, float_fmt="{:,.2f}", index_label=disp("gene"))


def ex11_render(cell_type, cytokine, padj_thr, lfc_thr, n_labels, genes_text):
    if cytokine not in DE.cytokines and DE.cached:
        msg('Elige una citocina de la lista.', "info"); return
    highlight, unknown = parse_genes(genes_text, 25)
    unknown_genes_message(unknown)
    if not DE.cached and (cell_type, cytokine) not in DE._memo:
        msg('⏳ No se encontraron los resultados precalculados: ejecutando DESeq2 para esta comparación (≈ 15 segundos)…', "info")
    res = DE.get(cell_type, cytokine)
    d = de_summary(res, padj_thr, lfc_thr)
    fig, ax = plt.subplots(figsize=(8.6, 6.4))
    draw_volcano(ax, d, padj_thr, lfc_thr, n_labels, highlight, f"{cyt_label(cytokine)} vs PBS · {CT_LABEL[cell_type]}")
    fig.tight_layout()
    show_fig(fig)
    if highlight:
        display(HTML(f'<div class=\"cx-label\">Tus genes</div>' + gene_stats_table(d, highlight, cell_type, cytokine, padj_thr, lfc_thr)))
    display(HTML('<div style="display:flex; flex-wrap:wrap; gap:28px"><div><div class="cx-label">Top 10 genes sobre-expresados <span class="cx-hint">(mayor evidencia × efecto)</span></div>'
                 + top_table(d, "up") + '</div><div><div class="cx-label">Top 10 genes infra-expresados</div>' + top_table(d, "down") + '</div></div>'))


def exercise_volcano():
    cyts = [c for c in adata.obs["cytokine"].cat.categories if c != "PBS"]
    c = dict(cell_type=dropdown([(CT_LABEL[x], x) for x in CELL_TYPES], "CD14_Mono", "190px"), cytokine=dropdown([(cyt_label(x), x) for x in cyts], "IFN-gamma", "190px"),
             padj_thr=widgets.SelectionSlider(options=PADJ_OPTIONS, value=PADJ_DEFAULT, continuous_update=False, layout=_lay("300px"), style=STYLE),
             lfc_thr=float_slider(0, 3, LFC_DEFAULT, 0.25, "270px", ".2f"), n_labels=int_slider(0, 30, 12, 2, "230px"), genes_text=None)
    gtext, gwidget = gene_picker("CXCL9, CXCL10, IDO1, STAT1, IRF1", "300px", 'p. ej. CXCL9, IDO1', label='Busca y resalta genes')
    c["genes_text"] = gtext
    rows = [[field('Tipo celular', c["cell_type"]), field('Citocina (vs PBS)', c["cytokine"]), field('Significación: p-valor ajustado menor que…', c["padj_thr"]),
             field('Tamaño del efecto: |log₂ fold change| mayor que…', c["lfc_thr"], "0.5 ≈ 1.4× · 1 = 2× · 2 = 4×"), field('Etiquetas de genes', c["n_labels"], 'los genes más fuertes')], [gwidget]]
    ex = Exercise(c, ex11_render, rows, title='¿Qué genes cambian con una citocina?')
    ex.show()
    return ex


# ══════════════════════════════════════════════════════════════════════════════════════════════════
#  Exercise 12: comparing two DE signatures (correlation of log2 fold changes)
# ══════════════════════════════════════════════════════════════════════════════════════════════════
GENE_SUBSETS = [('todos los genes analizados en ambas comparaciones', "all"), ('genes significativos en A o en B', "either"), ('genes significativos en A y en B', "both")]
EXAMPLES12 = [('— elige un ejemplo —', ""),
              ('IL-4 vs IL-13 en monocitos: casi la misma respuesta', ("CD14_Mono", "IL-4", "CD14_Mono", "IL-13")),
              ('IL-4 vs IL-13 en células T CD4: mucho menos parecida', ("CD4_T_cell", "IL-4", "CD4_T_cell", "IL-13")),
              ('IFN-β vs IFN-γ en monocitos: dos interferones', ("CD14_Mono", "IFN-beta", "CD14_Mono", "IFN-gamma")),
              ('IFN-γ vs IL-4 en monocitos: programas opuestos', ("CD14_Mono", "IFN-gamma", "CD14_Mono", "IL-4")),
              ('TNF-α en monocitos vs en células T CD4: mismo estímulo, dos tipos celulares', ("CD14_Mono", "TNF-alpha", "CD4_T_cell", "TNF-alpha")),
              ('IL-6 en monocitos vs en células T CD4: mismo estímulo, dos tipos celulares', ("CD14_Mono", "IL-6", "CD4_T_cell", "IL-6")),
              ('TNF-α vs IL-6 en monocitos', ("CD14_Mono", "TNF-alpha", "CD14_Mono", "IL-6")),
              ('IL-2 vs IL-15 en células NK: el mismo receptor', ("NK_CD56low", "IL-2", "NK_CD56low", "IL-15"))]
COL_A, COL_B, COL_BOTH = BLUE, ORANGE, PURPLE


def compare_tables(da, db, subset, padj_thr, lfc_thr):
    """Merge two DE tables on the genes tested in both and compute the statistics of the comparison."""
    ok = da["tested"] & db["tested"]
    m = pd.DataFrame({"a": da.loc[ok, "log2fc"], "b": db.loc[ok, "log2fc"], "dir_a": da.loc[ok, "direction"], "dir_b": db.loc[ok, "direction"]})
    sa, sb = m["dir_a"] != "ns", m["dir_b"] != "ns"
    m["cls"] = np.select([sa & sb, sa, sb], ["both", "A", "B"], "none")
    use = {"all": np.ones(len(m), bool), "either": (sa | sb).values, "both": (sa & sb).values}[subset]
    return m, use


def correlations(m, use):
    x, y = m.loc[use, "a"].values.astype(float), m.loc[use, "b"].values.astype(float)
    if len(x) < 5 or np.std(x) == 0 or np.std(y) == 0:
        return float("nan"), float("nan"), len(x)
    return float(np.corrcoef(x, y)[0, 1]), float(stats.spearmanr(x, y)[0]), len(x)


def similarity_ranking(ct, cyt, subset_all=True):
    """Pearson correlation of the log2FC of (ct, cyt) with every other cytokine in the same cell type (cache only)."""
    i, j = DE.cell_types.index(ct), DE.cytokines.index(cyt)
    lfc, nlp = DE.arrays["lfc"][i].astype(np.float32), DE.arrays["nlp_adj"][i].astype(np.float32)
    out = {}
    for k, name in enumerate(DE.cytokines):
        if k == j:
            continue
        ok = ~np.isnan(nlp[j]) & ~np.isnan(nlp[k])
        if ok.sum() > 50:
            out[name] = float(np.corrcoef(lfc[j][ok], lfc[k][ok])[0, 1])
    return pd.Series(out).sort_values(ascending=False)


def ex12_render(cell_a, cyt_a, cell_b, cyt_b, subset, padj_thr, lfc_thr, genes_text, label_top):
    if cyt_a == cyt_b and cell_a == cell_b:
        msg('A y B son la <b>misma comparación</b>: la correlación es trivialmente 1. Cambia la citocina o el tipo celular de B.', "info")
    if not DE.cached and any((c, y) not in DE._memo for c, y in ((cell_a, cyt_a), (cell_b, cyt_b))):
        msg('⏳ No se encontraron los resultados precalculados: ejecutando DESeq2 (≈ 15 segundos por comparación)…', "info")
    da, db = de_summary(DE.get(cell_a, cyt_a), padj_thr, lfc_thr), de_summary(DE.get(cell_b, cyt_b), padj_thr, lfc_thr)
    m, use = compare_tables(da, db, subset, padj_thr, lfc_thr)
    highlight, unknown = parse_genes(genes_text, 25)
    unknown_genes_message(unknown)
    r, rho, n = correlations(m, use)
    na, nb = f"{cyt_label(cyt_a)} · {CT_SHORT[cell_a]}", f"{cyt_label(cyt_b)} · {CT_SHORT[cell_b]}"
    ranking = similarity_ranking(cell_a, cyt_a) if DE.cached else None
    fig = plt.figure(figsize=(13, 6.0))
    gs = fig.add_gridspec(1, 2, width_ratios=[1.35, 1], wspace=0.35)
    ax = fig.add_subplot(gs[0])
    for cls, col, lab, s_ in (("none", NS_COLOR, 'ninguno', 5), ("A", COL_A, 'solo A', 11), ("B", COL_B, 'solo B', 11), ("both", COL_BOTH, "both", 13)):
        mm = m[m["cls"] == cls]
        ax.scatter(mm["a"], mm["b"], s=s_, alpha=0.55 if cls == "none" else 0.8, color=col, linewidths=0, label=f"{disp(lab)}  ({len(mm):,})", rasterized=True, zorder=2)
    ax.axhline(0, color="k", lw=0.6, zorder=1); ax.axvline(0, color="k", lw=0.6, zorder=1)
    lim = float(np.nanpercentile(np.abs(np.r_[m["a"].values, m["b"].values]), 99.8)) * 1.1
    ax.plot([-lim, lim], [-lim, lim], color="#999", lw=0.9, ls="--", zorder=1)
    sel = m[use]
    if len(sel) > 5 and np.std(sel["a"]) > 0:
        slope, icpt = np.polyfit(sel["a"].astype(float), sel["b"].astype(float), 1)
        xs = np.array([-lim, lim]); ax.plot(xs, icpt + slope * xs, color="k", lw=1.8, zorder=4)
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    texts = []
    if label_top and len(sel):
        mn = np.minimum(sel["a"].abs(), sel["b"].abs())
        same, opp = sel[np.sign(sel["a"]) == np.sign(sel["b"])], sel[np.sign(sel["a"]) != np.sign(sel["b"])]
        for part, k in ((same, 5), (opp, 3)):
            for g in mn[part.index].nlargest(k).index:
                if g not in highlight:
                    texts.append(ax.text(m.loc[g, "a"], m.loc[g, "b"], g, fontsize=7.5, fontstyle="italic", zorder=5))
    hl = [g for g in highlight if g in m.index]
    if hl:
        ax.scatter(m.loc[hl, "a"], m.loc[hl, "b"], s=70, facecolors="none", edgecolors="k", linewidths=1.6, zorder=6)
        for g in hl:
            texts.append(ax.text(m.loc[g, "a"], m.loc[g, "b"], g, fontsize=9.5, fontweight="bold", fontstyle="italic", zorder=7, bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="k", lw=0.6, alpha=0.9)))
    if HAS_ADJUSTTEXT and texts:
        try:
            adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle="-", color="#888", lw=0.5), expand=(1.2, 1.4), max_move=(30, 30))
        except Exception:
            pass
    ax.legend(loc="lower right", markerscale=2.2, fontsize=8.5, title='significativo en…', title_fontsize=8.5)
    ax.set_xlabel(f"log₂ fold change de A:  {cyt_label(cyt_a)} vs PBS en {CT_LABEL[cell_a]}"); ax.set_ylabel(f"log₂ fold change de B:  {cyt_label(cyt_b)} vs PBS en {CT_LABEL[cell_b]}")
    ax.set_title(f"Pearson r = {r:.2f}   ·   Spearman ρ = {rho:.2f}   ·   {n:,} genes", fontsize=12)
    ax2 = fig.add_subplot(gs[1])
    if ranking is not None and cell_a == cell_b and cyt_b in ranking.index:
        sel_r = pd.concat([ranking.head(8), ranking.tail(4)])
        if cyt_b not in sel_r.index:
            sel_r = pd.concat([ranking.head(8), ranking.loc[[cyt_b]], ranking.tail(3)])
        cols = [COL_B if k == cyt_b else (UP_COLOR if v > 0 else DOWN_COLOR) for k, v in sel_r.items()]
        ax2.barh([cyt_label(k) for k in sel_r.index], sel_r.values, color=cols)
        ax2.axvline(0, color="k", lw=0.6); ax2.invert_yaxis()
        ax2.set_xlabel('correlación de los log₂ fold change con A'); ax2.set_title(f"En {CT_LABEL[cell_a]}, ¿quién responde como {cyt_label(cyt_a)}?\n(B = {cyt_label(cyt_b)} en naranja; puesto {int((ranking > ranking[cyt_b]).sum()) + 1} de {len(ranking)})", fontsize=10)
        ax2.tick_params(axis="y", labelsize=8.5)
    elif ranking is not None:
        sel_r = pd.concat([ranking.head(8), ranking.tail(4)])
        ax2.barh([cyt_label(k) for k in sel_r.index], sel_r.values, color=[UP_COLOR if v > 0 else DOWN_COLOR for v in sel_r.values])
        ax2.axvline(0, color="k", lw=0.6); ax2.invert_yaxis(); ax2.tick_params(axis="y", labelsize=8.5)
        ax2.set_xlabel('correlación de los log₂ fold change con A'); ax2.set_title(f"En {CT_LABEL[cell_a]}, ¿quién responde como {cyt_label(cyt_a)}?", fontsize=10)
    else:
        ax2.axis("off")
    fig.tight_layout()
    show_fig(fig)
    # which genes explain the (dis)similarity?
    mn = np.minimum(sel["a"].abs(), sel["b"].abs()) if len(sel) else pd.Series(dtype=float)
    def gene_tab(part, k=8):
        top = mn[part.index].nlargest(k).index
        df = pd.DataFrame({'log₂FC en A': m.loc[top, "a"], 'log₂FC en B': m.loc[top, "b"]})
        return html_table(df, index_label="gene")
    if len(sel):
        same_up = sel[(sel["a"] > 0) & (sel["b"] > 0)]; same_dn = sel[(sel["a"] < 0) & (sel["b"] < 0)]; opp = sel[np.sign(sel["a"]) != np.sign(sel["b"])]
        display(HTML(
            f'''<div class=\"cx-info\">Entre los {n:,} genes usados: <b>{len(same_up):,}</b> suben en ambas, <b>{len(same_dn):,}</b> bajan en ambas, <b>{len(opp):,}</b> cambian en sentido opuesto. Pearson r = <b>{r:.2f}</b> (Spearman ρ = {rho:.2f}).</div><div style=\"display:flex; flex-wrap:wrap; gap:28px\"><div><div class=\"cx-label\">Suben en ambas (mayor cambio compartido)</div>{gene_tab(same_up)}</div><div><div class=\"cx-label\">Bajan en ambas</div>{gene_tab(same_dn)}</div><div><div class=\"cx-label\">Sentidos opuestos</div>{gene_tab(opp, 6)}</div></div>'''))
    else:
        msg('Ningún gen cumple el filtro: prueba <i>todos los genes analizados en ambas comparaciones</i> o relaja los umbrales.', "info")
    if hl:
        df = pd.DataFrame({'log₂FC en A': m.loc[hl, "a"], 'p aj. en A': [f"{da.loc[g, 'padj']:.1e}" for g in hl], 'log₂FC en B': m.loc[hl, "b"], 'p aj. en B': [f"{db.loc[g, 'padj']:.1e}" for g in hl]}, index=hl)
        display(HTML('<div class="cx-label" style="margin-top:8px">Tus genes</div>' + html_table(df, index_label="gene")))
    missing = [g for g in highlight if g not in m.index]
    if missing:
        msg('No analizados en ambas comparaciones (fuera de la lista de genes inmunes, o muy pocos conteos): ' + ", ".join(missing), "warn")


def exercise_compare():
    cyts = [c for c in adata.obs["cytokine"].cat.categories if c != "PBS"]
    cts = [(CT_LABEL[x], x) for x in CELL_TYPES]
    cyt_opts = [(cyt_label(x), x) for x in cyts]
    c = dict(cell_a=dropdown(cts, "CD14_Mono", "170px"), cyt_a=dropdown(cyt_opts, "IL-4", "170px"), cell_b=dropdown(cts, "CD14_Mono", "170px"), cyt_b=dropdown(cyt_opts, "IL-13", "170px"),
             subset=dropdown(GENE_SUBSETS, "all", "290px"),
             padj_thr=widgets.SelectionSlider(options=PADJ_OPTIONS, value=PADJ_DEFAULT, continuous_update=False, layout=_lay("260px"), style=STYLE),
             lfc_thr=float_slider(0, 3, LFC_DEFAULT, 0.25, "230px", ".2f"), genes_text=None, label_top=checkbox('etiquetar los genes que determinan el patrón', True, "300px"))
    gtext, gwidget = gene_picker("", "280px", 'p. ej. STAT1, CCL17', with_sets=False, label='Resalta genes (opcional)')
    c["genes_text"] = gtext
    example = dropdown(EXAMPLES12, "", "480px")
    ex = None

    def _load(change):
        v = change["new"]
        if v:
            ex.set_values(cell_a=v[0], cyt_a=v[1], cell_b=v[2], cyt_b=v[3])
            example.value = ""
    example.observe(_load, "value")
    rows = [[field('Carga un ejemplo', example, '…o elige A y B tú mismo abajo')],
            [widgets.VBox([widgets.HTML('<div class="cx-label" style="color:%s">COMPARACIÓN A</div>' % COL_A), widgets.HBox([field('Tipo celular', c["cell_a"]), field('Citocina vs PBS', c["cyt_a"])])]),
             widgets.VBox([widgets.HTML('<div class="cx-label" style="color:%s">COMPARACIÓN B</div>' % COL_B), widgets.HBox([field('Tipo celular', c["cell_b"]), field('Citocina vs PBS', c["cyt_b"])])])],
            [field('Genes usados para la correlación', c["subset"]), field('«Significativo» = p ajustado menor que…', c["padj_thr"]), field('…y |log₂FC| mayor que', c["lfc_thr"]), gwidget, c["label_top"]]]
    ex = Exercise(c, ex12_render, rows, title='¿Dos estímulos activan los mismos genes?')
    ex.show()
    return ex


# ══════════════════════════════════════════════════════════════════════════════════════════════════
#  90_init.py: everything slow happens here, ONCE (load the data, normalise, PCA, load the DESeq2 results)
# ══════════════════════════════════════════════════════════════════════════════════════════════════
GENESETS_WANTED = {
    'Marcadores de monocitos': ["LYZ", "CD14", "S100A9", "FCN1"],
    'Marcadores de células T': ["CD3E", "CD3D"],
    'Marcadores de células T CD8': ["CD8A", "CD8B"],
    'Marcadores de células NK': ["NKG7", "GNLY", "KLRF1"],
    'Respuesta a IFN-γ / STAT1': ["STAT1", "IRF1", "GBP1", "IDO1", "CXCL9", "CXCL10"],
    'Respuesta a interferón tipo I': ["ISG15", "IFIT1", "IFIT3", "MX1", "OAS1"],
    'Respuesta a IL-4 / STAT6': ["ALOX15", "CCL17", "FCER2"],
    'Respuesta a IL-1β / NF-κB': ["IL6", "CXCL8", "CCL2"],
    'Respuesta a IL-2 / STAT5': ["IL2RA", "GZMB", "CISH"],
    'Genes constitutivos (housekeeping)': ["ACTB", "GAPDH", "B2M", "TMSB4X", "UBC"],
}

adata = None          # the AnnData object of the practical (X = raw counts; layers: cpm, log2cpm)
EXPRESSED = None      # True for the genes that pass the expression filter (≥ 10 counts in ≥ 12 samples)
PCA_ALL, PCA_CT, DE = None, {}, None


def init_practical(h5ad_path=None, de_cache_path=None, verbose=True, with_de=True):
    """Load the data and pre-compute everything the exercises need. Takes ~30 s once the files are on disk."""
    global adata, EXPRESSED, PCA_ALL, PCA_CT, DE, GENE_NAMES, GENE_LOOKUP, GENESETS
    t0 = time.time()
    say = (lambda s: print(s, flush=True)) if verbose else (lambda s: None)
    path = get_h5ad(h5ad_path, verbose)
    say('📖 Leyendo y ordenando los datos …')
    adata = load_adata(path)
    EXPRESSED = is_expressed(adata.X)
    GENE_NAMES = list(adata.var_names)
    GENE_LOOKUP = {}
    for g in GENE_NAMES:
        GENE_LOOKUP.setdefault(g.upper(), g)
    GENESETS = {k: [g for g in v if g in GENE_LOOKUP.values()] for k, v in GENESETS_WANTED.items()}
    say('🧮 Calculando los PCA (todas las muestras, y uno por tipo celular) …')
    PCA_ALL = make_pca(adata, expressed=EXPRESSED)
    PCA_CT = {ct: make_pca(adata, mask=(adata.obs["cell_type"] == ct).values, expressed=EXPRESSED) for ct in CELL_TYPES}
    if with_de:
        say('📦 Cargando los resultados de expresión diferencial …')
        DE = DEResults(adata, de_cache_path or get_de_cache(verbose), verbose)
    say(f"✅ Listo en {time.time() - t0:.0f} s: {adata.n_obs:,} muestras × {adata.n_vars:,} genes. Baja y empieza con el Ejercicio 1.")
    return adata

DISPLAY.update({'both': 'ambas', 'gene': 'gen', 'control': 'control', 'treated': 'tratado', 'condition': 'condición', 'donor': 'donante', 'cytokine': 'citocina', 'Donor': 'Donante'})


# ── Load the data and pre-compute everything ──────────────────────────────────────────────────────
init_practical()


---
# Parte A · Calentamiento en Python
Dos ejercicios cortos que no necesitan datos: qué es una **variable** y cómo **mirar números** con gráficos.

## Ejercicio 1 · Variables: ponerle nombre a los valores
⏱ *5 min*

Una **variable** es un **nombre** que guarda un **valor**, como un tubo etiquetado en el congelador. Se crea con `=`: el nombre a la izquierda, el valor a la derecha.
Python necesita saber *de qué tipo* es el valor: **texto** (`str`, siempre entre comillas), **números enteros** (`int`), **números decimales** (`float`), **Verdadero/Falso** (`bool`) o una **lista** de valores.
`print(...)` muestra un valor en pantalla, y las variables se pueden combinar para crear otras nuevas.

**Qué hacer:** cambia los controles y observa cómo cambian el código Python, lo que se imprime y la tabla de variables.

In [ ]:
#@title ▶ Ejercicio 1 · Variables { display-mode: "form" }
try:
    exercise_variables()
except NameError:
    print('⚠️ Ejecuta primero la celda de Preparación al principio del cuaderno (▶), espera a ✅ Listo y vuelve a ejecutar esta celda.')


#### ❓ Preguntas
1. Cambia `message` por tu nombre. ¿Qué líneas del panel de código y de la salida cambian?
2. Mueve el deslizador `n_donors`. ¿Qué líneas impresas cambian y cuáles no? ¿Por qué?
3. En la tabla de variables, ¿de qué tipo es `library_size`? ¿Por qué no es un número entero?

<details>
<summary>💡 Respuestas</summary>

1. La línea `message = '...'` y la primera línea impresa. Los textos se escriben **entre comillas**; la variable guarda lo que escribas.
2. Solo las líneas que *usan* `n_donors`: `Donantes: …` y `… se añadió a … donantes`. Cambiar una variable solo afecta a los sitios donde aparece su nombre.
3. Es un `float` (decimal): `library_size = n_cells * umis_per_cell` multiplica un entero por un decimal y el resultado es decimal. Si ambos fueran enteros (`int`), el resultado sería `int`. (`library_size`, el número total de conteos de una muestra, volverá a aparecer en el Ejercicio 3.)

</details>

## Ejercicio 2 · Mirar números: gráficos básicos
⏱ *7 min*

Un gráfico convierte muchos números en una imagen. Cada tipo de gráfico responde a una pregunta distinta:

| Gráfico | Responde a la pregunta… |
|---|---|
| 📊 **Histograma** | *¿Cómo se distribuyen los valores?* (cuántos donantes tienen cada valor) |
| 📦 **Diagrama de cajas** | *¿Dónde está el centro y cuánto se dispersan los valores?* Línea = mediana, caja = el 50 % central, bigotes = el rango |
| 📶 **Barras** | *¿Cuál es la media de cada grupo?* (¡oculta los valores individuales!) |
| 🔵 **Dispersión** | *¿Están relacionadas dos medidas?* Un punto por donante |
| 📈 **Líneas** | *¿Cómo cambia un valor con el tiempo?* |

**Qué hacer:** los datos son **simulados**: un gen medido en varios donantes, sin (control) y con (tratado) una citocina. Elige un tipo de gráfico y juega con los números.

In [ ]:
#@title ▶ Ejercicio 2 · Gráficos básicos { display-mode: "form" }
try:
    exercise_plotting()
except NameError:
    print('⚠️ Ejecuta primero la celda de Preparación al principio del cuaderno (▶), espera a ✅ Listo y vuelve a ejecutar esta celda.')


#### ❓ Preguntas
1. En el **histograma**, pon el efecto de la citocina a 0 y luego a +2. ¿Qué se mueve? ¿Qué pasa si además aumentas la variabilidad entre donantes (DE)?
2. Usa pocos donantes (5–10) y pulsa **🎲 nuevo sorteo aleatorio** varias veces (histograma o cajas). ¿Los dos grupos se ven siempre igual? ¿Qué te dice esto de los experimentos con pocos donantes?
3. En el **diagrama de dispersión**, usa un efecto de +1 y compara un parecido de 0.1 con otro de 0.95. ¿Cuándo se ve mejor el efecto? ¿Por qué ayudaría comparar cada donante *consigo mismo*?

<details>
<summary>💡 Respuestas</summary>

1. Con efecto 0 los dos histogramas se solapan por completo; con +2 el tratado se desplaza 2 unidades a la derecha (2 unidades de log₂ = **4 veces** más expresión). Una DE mayor ensancha ambos histogramas, que se solapan más aunque el efecto sea el mismo: el efecto es más difícil de ver cuando los donantes difieren mucho.
2. No: con pocos donantes el azar cambia las medias y las formas, y puedes ver "diferencias" (o perder reales) por mala suerte. Con más donantes las imágenes son estables. Por eso los experimentos necesitan réplicas.
3. Con parecido 0.95 todos los puntos quedan en una banda estrecha **por encima** de la línea discontinua (`y = x`): *cada* donante sube más o menos lo mismo, aunque partan de niveles muy distintos. Con 0.1 la nube es redonda y el desplazamiento queda tapado por las diferencias entre donantes. Comparar cada donante consigo mismo (un diseño **pareado**, Ejercicios 11–12) elimina las diferencias entre personas.

</details>

---
# Parte B · Conoce los datos

## Ejercicio 3 · El objeto AnnData: muestras, genes y sus metadatos
⏱ *10 min*

Nuestros datos viven en un objeto **AnnData** llamado `adata`, que mantiene varias tablas **alineadas**:

| Componente | Qué contiene | Forma |
|---|---|---|
| `X` | los **números**: conteos crudos (cuántas moléculas de mRNA de cada gen se detectaron) | muestras × genes |
| `obs` | información de las **muestras** (filas de `X`): donante, citocina, tipo celular… | una fila por muestra |
| `var` | información de los **genes** (columnas de `X`) | una fila por gen |
| `layers` | otras versiones de `X` con la misma forma (aquí, versiones *normalizadas* que construiremos en los Ejercicios 5–6) | muestras × genes |

El objeto completo tiene **4,224 muestras × 18,120 genes**: 12 donantes × 4 tipos celulares × 88 condiciones (87 citocinas + PBS).
Una **muestra** es un pseudobulk: todas las células de un donante, un tipo celular y una condición, sumadas. Su nombre es `donante|tipo celular|citocina`.
Seleccionar funciona como en una tabla: `adata[muestras, genes]`, **primero muestras, después genes**.

**Qué hacer:** elige muestras y genes (por defecto: los 4 controles PBS del Donor 1, más sus monocitos tratados con IFN-γ). El **subconjunto** que construyes aparece en las tres pestañas, y su lugar en la matriz completa, a la derecha.

In [ ]:
#@title ▶ Ejercicio 3 · El objeto AnnData { display-mode: "form" }
try:
    exercise_anndata()
except NameError:
    print('⚠️ Ejecuta primero la celda de Preparación al principio del cuaderno (▶), espera a ✅ Listo y vuelve a ejecutar esta celda.')


#### ❓ Preguntas
1. En el subconjunto por defecto, ¿qué muestra tiene el mayor conteo de `LYZ`, de `CD3E` y de `NKG7`? ¿Concuerda con lo que sabes de estos genes?
2. Abre la pestaña **obs**: ¿qué te dicen `psbulk_n_cells` y `library_size`? Compara las dos muestras de monocitos (PBS e IFN-γ).
3. `STAT1` se activa con IFN-γ. Compara su conteo en las dos muestras de monocitos. ¿Es mayor con IFN-γ? ¿Es lo que esperabas?

<details>
<summary>💡 Respuestas</summary>

1. `LYZ` es máximo en los **monocitos** (911,945 conteos), `CD3E` en las **células T CD4** (20,361; las T CD8 son segundas con 3,100) y `NKG7` en las **células NK** (2,392; luego las T CD8, también citotóxicas): son marcadores de tipo celular.
2. `psbulk_n_cells` es el **número de células que se sumaron** en la muestra y `library_size` el **número total de conteos** (de todos los genes). La muestra PBS de monocitos contiene 17,373 células y 147.9 millones de conteos; la de IFN-γ solo 2,556 células y 21.4 millones: unas **7 veces menos material**. (El control PBS es la suma de 6 pocillos.)
3. **No: es ~9 veces *menor* con IFN-γ** (73,853 frente a 639,289), aunque IFN-γ induce `STAT1`. El conteo crudo mezcla **biología** (¿está el gen activo?) con la **cantidad de material** (cuántas células, qué profundidad). Es el problema que resolvemos en la Parte C.

</details>

## Ejercicio 4 · Distribuciones de conteos crudos: diagramas de cajas
⏱ *8 min*

Un **diagrama de cajas** resume muchos valores: la **línea** es la mediana, la **caja** contiene el 50 % central de los valores y los **bigotes** muestran el rango. Cada **punto** es una **muestra pseudobulk**
(un donante con una citocina en un tipo celular); la *caja* es, por tanto, la distribución del gen en todas las muestras de un grupo.
El panel derecho muestra el **tamaño de librería** (conteos totales) de las mismas muestras.

**Qué hacer:** por defecto verás cuatro genes con distintos papeles en los cuatro tipos celulares (las 88 condiciones, 1,056 muestras por tipo celular). Cambia los genes, los grupos o los tipos celulares.
Prueba "control (PBS) vs tratado" para comparar los dos tipos de muestras.

In [ ]:
#@title ▶ Ejercicio 4 · Diagramas de cajas de conteos crudos { display-mode: "form" }
try:
    exercise_boxplots()
except NameError:
    print('⚠️ Ejecuta primero la celda de Preparación al principio del cuaderno (▶), espera a ✅ Listo y vuelve a ejecutar esta celda.')


#### ❓ Preguntas
1. En el gráfico por defecto, ¿cuántos órdenes de magnitud separan el gen más y el menos abundante en monocitos? ¿Se comportan `LYZ`, `CD3E` y `NKG7` como marcadores de tipo celular?
2. `ACTB` es un gen constitutivo (siempre activo), pero sus conteos difieren entre tipos celulares. Compara con el panel derecho: ¿hay relación?
3. Agrupa las cajas por **control (PBS) vs tratado**. ¿Qué ves para *todos* los genes? ¿Por qué?

<details>
<summary>💡 Respuestas</summary>

1. En monocitos la mediana va de 4 (`NKG7`) a 25,725 (`STAT1`): **casi 4 órdenes de magnitud**. `LYZ` es máximo en monocitos (mediana 30,058 frente a ≤ 297 en el resto), `CD3E` en células T CD4 y CD8 (1,262 y 663) y `NKG7` en NK (134; luego T CD8, 65).
2. Sí: el **tamaño de librería** difiere entre tipos celulares (medianas: NK 1.2 M, T CD8 3.8 M, monocitos 7.6 M, T CD4 8.7 M) y los conteos de `ACTB` lo siguen en buena parte (NK el menor, luego T CD8). Los monocitos son células más grandes con más mRNA, así que tienen aún más `ACTB` por conteo. Los conteos dependen de cuánto material aporta cada tipo celular.
3. Los **controles PBS son más altos en todos los genes**, incluido `ACTB`: su tamaño de librería es ~5 veces mayor (mediana 23 M frente a 4.4 M) porque cada muestra PBS es la suma de 6 pocillos. Esa diferencia no tiene que ver con la biología de cada gen, sino con la cantidad de material, que corregimos en el siguiente ejercicio.

</details>

---
# Parte C · Normalización: hacer comparables las muestras

## Ejercicio 5 · Conteos crudos vs CPM: ¿por qué normalizamos?
⏱ *10 min*

Un conteo crudo depende de **dos cosas**: cuánto se expresa el gen (biología) y cuánto material tenemos (cuántas células, qué profundidad de secuenciación; técnica).
El **tamaño de librería** es el número total de conteos de una muestra. Para quitar el efecto de la profundidad expresamos cada gen como fracción de su muestra:

**CPM** (conteos por millón) = conteo del gen ÷ tamaño de librería de la muestra × 1,000,000

**Qué hacer:**
* **Ejemplo de juguete** (simulado): el mismo donante con una muestra profunda (PBS) y otra superficial (con una citocina). Solo el gen B cambia de verdad. Mueve los deslizadores.
* **Datos reales, PBS vs citocina**: una línea por donante. Por defecto, el gen constitutivo `ACTB` en monocitos con IFN-γ. Cambia el gen, el tipo celular o la citocina.
* **Datos reales, todas las muestras**: ¿sigue el conteo del gen al tamaño de librería?

In [ ]:
#@title ▶ Ejercicio 5 · Conteos crudos vs CPM { display-mode: "form" }
try:
    exercise_normalization()
except NameError:
    print('⚠️ Ejecuta primero la celda de Preparación al principio del cuaderno (▶), espera a ✅ Listo y vuelve a ejecutar esta celda.')


#### ❓ Preguntas
1. **Ejemplo de juguete**: con una profundidad del 20 % y un cambio real de ×4, ¿qué dicen los conteos crudos del gen B? ¿Y el CPM? ¿Y con profundidad del 100 %?
2. **Datos reales**, `ACTB` en monocitos (PBS vs IFN-γ): ¿cómo cambian el tamaño de librería, el conteo crudo de `ACTB` y su CPM? ¿Qué esperas de un gen constitutivo?
3. Cambia el gen a `IDO1` y luego a `STAT1` (ambos inducidos por IFN-γ). ¿Qué dicen conteos crudos y CPM? Después mira la vista **todas las muestras**: ¿cómo cambia la correlación con el tamaño de librería de crudos a CPM?

<details>
<summary>💡 Respuestas</summary>

1. **Conteos crudos: el gen B parece ligeramente *a la baja* (×0.78)** y todos los demás genes parecen ~5 veces más bajos, solo porque la muestra 2 tiene 5 veces menos profundidad. **CPM: los genes A, C y D no cambian (×0.97) y el gen B es ×3.9**, cerca del ×4 real. Con profundidad del 100 % ambas representaciones coinciden: normalizar solo importa cuando las profundidades difieren.
2. El tamaño de librería es **6 veces menor** con IFN-γ (×0.16) y el conteo crudo de `ACTB` lo sigue (×0.16); tras el CPM `ACTB` **no cambia (×0.95)**, como se espera de un gen constitutivo. En la vista de todas las muestras la correlación de `ACTB` con el tamaño de librería es **r = +0.84 en crudos y −0.25 en CPM**: el efecto de la profundidad desaparece.
3. `IDO1`: los conteos crudos dicen **sin cambio (×1.1)**, el CPM dice **×6.7 al alza**. `STAT1`: los crudos dicen **×4.5 *a la baja***, el CPM **×1.35 al alza**. Sin normalizar perderíamos la inducción de `IDO1` y concluiríamos lo contrario para `STAT1`.

</details>

## Ejercicio 6 · El logaritmo: domar la escala
⏱ *8 min*

El CPM arregla las diferencias entre **muestras**, pero no entre **genes**: unos tienen <1 CPM y otros >10,000, los valores son muy **asimétricos** y la **varianza crece con la media**.
Muchos análisis (PCA, tests…) se guían por la varianza, así que unos pocos genes abundantes lo decidirían todo. El **logaritmo** comprime los valores grandes y convierte cocientes en diferencias
(con log₂: duplicar = +1, reducir a la mitad = −1). Como log(0) no existe, primero sumamos un pequeño **pseudoconteo** (normalmente 1): **log₂(CPM + 1)**.

**Qué hacer:** los paneles superiores muestran un gen en un tipo celular; los inferiores, **todos los genes** (varianza frente a media). Cambia la **base** del logaritmo y el **pseudoconteo**.

In [ ]:
#@title ▶ Ejercicio 6 · El logaritmo { display-mode: "form" }
try:
    exercise_logtransform()
except NameError:
    print('⚠️ Ejecuta primero la celda de Preparación al principio del cuaderno (▶), espera a ✅ Listo y vuelve a ejecutar esta celda.')


#### ❓ Preguntas
1. Compara ① y ② para `CXCL10` en monocitos. ¿Qué le pasa a la forma de la distribución y a su varianza?
2. Compara ③ y ④: ¿qué parte de la varianza total acumulan los 20 genes más variables, antes y después del logaritmo? ¿Qué significaría eso para un PCA?
3. Cambia la base a 10 y a *e*. ¿Qué cambia y qué no? ¿Y con el pseudoconteo a 0 y a 10?

<details>
<summary>💡 Respuestas</summary>

1. En CPM la distribución es **asimétrica** (cola larga: media 1,297, mediana 634, asimetría 1.6) y su varianza enorme (2,350,366). Tras log₂ es mucho más simétrica (asimetría −0.3) y la varianza es 4.7.
2. En CPM **20 genes acumulan el 41 % de toda la varianza**; tras log₂(CPM + 1) solo el **3 %**. Sin logaritmo, un PCA lo decidirían unos pocos genes abundantes (y casi solo "vería" esos); con él contribuyen miles de genes.
3. La **forma no cambia**, solo los números del eje (log₂ x = 3.32 × log₁₀ x): las varianzas se multiplican por 0.48 (ln) o 0.09 (log₁₀). La base es una convención; usamos 2 porque una diferencia de 1 significa "el doble". Con pseudoconteo **0**, los ceros no se pueden transformar y se descartan. Con **10**, los valores bajos se aplastan. 1 es el compromiso habitual.

</details>

---
# Parte D · Buscar patrones en los datos
A partir de ahora usamos los valores **normalizados**: log₂(CPM + 1).

## Ejercicio 7 · Mapas de calor: genes × muestras
⏱ *10 min*

Un **mapa de calor** muestra muchos genes (filas) en muchas muestras (columnas) a la vez. Para comparar patrones, cada gen (fila) se **escala entre las muestras** (z-score): **rojo** = por encima de la media de *ese* gen, **azul** = por debajo.
Las dos barras superiores muestran el **tipo celular** y el **tamaño de librería** de cada columna (oscuro = profundo).

**Qué hacer:** por defecto verás marcadores de tipo celular y genes de respuesta a citocinas en los cuatro tipos celulares con PBS y cuatro citocinas (12 donantes cada una), con los **conteos crudos** a la izquierda y los **valores normalizados** a la derecha.
Cambia de vista, elige otros **grupos de genes**, añade tus propios genes (p. ej. `TNF`, `IL1B`, `IL13RA1`) y selecciona otras **citocinas** (Ctrl/⌘ + clic).

In [ ]:
#@title ▶ Ejercicio 7 · Mapas de calor { display-mode: "form" }
try:
    exercise_heatmaps()
except NameError:
    print('⚠️ Ejecuta primero la celda de Preparación al principio del cuaderno (▶), espera a ✅ Listo y vuelve a ejecutar esta celda.')


#### ❓ Preguntas
1. Compara ANTES y DESPUÉS. ¿Qué columnas son las más brillantes en el mapa crudo, para casi todos los genes? ¿Puedes leer los tipos celulares en cada panel?
2. En el mapa normalizado, ¿qué citocina activa qué bloque de genes, y en qué tipo celular?
3. Selecciona **IFN-beta** y el grupo *Respuesta a interferón tipo I*. ¿En qué tipos celulares responden estos genes? Después añade `IL13RA1` e `IL4R` como genes extra: ¿qué tipo celular expresa `IL13RA1`? (tenlo presente para el Ejercicio 12)

<details>
<summary>💡 Respuestas</summary>

1. **Antes**, las columnas más brillantes son las **PBS** (oscuras en la barra de tamaño de librería), sea cual sea el gen: la imagen muestra *cuántos conteos tiene cada muestra* y los tipos celulares cuesta leerlos. **Después**, el brillo de PBS desaparece y cada tipo celular muestra sus marcadores: `LYZ`, `CD14`, `S100A9`, `FCN1` en monocitos; `CD3E`, `CD3D` en células T; `CD8A`, `CD8B` en T CD8; `NKG7`, `GNLY`, `KLRF1` en NK.
2. **IL-4** → `ALOX15`, `CCL17`, `FCER2` casi solo en **monocitos**. **IFN-γ** → `GBP1`, `IDO1`, `CXCL9`, `CXCL10`, máximo en **monocitos**. **IL-1β** → `IL6`, `CXCL8`, `CCL2` en **monocitos**. **IL-2** → `IL2RA`, `GZMB`, `CISH` en **NK y T**, los linfocitos que tienen el receptor de IL-2.
3. Los genes estimulados por interferón (`ISG15`, `IFIT1`, `MX1`…) suben **en los cuatro tipos celulares** con IFN-β (log₂ fold change entre 2 y 3), porque el receptor de interferón tipo I está en todos. `IL13RA1` (una subunidad del receptor de IL-13) se expresa en **monocitos** (~256 CPM) y prácticamente no en T y NK (~2 CPM).

</details>

> ⚠️ **Los z-score son para visualizar.** Ocultan el nivel absoluto de cada gen y exageran genes que apenas cambian. Nunca calcules fold changes a partir de ellos.

## Ejercicio 8 · PCA: la idea, con solo dos genes
⏱ *10 min*

El **PCA** (análisis de componentes principales) resume muchos genes en unos pocos ejes nuevos. Para ver cómo, toma solo **dos genes**: cada muestra es un punto en un plano (*x* = gen 1, *y* = gen 2).
1. Dibuja un **eje** (una recta por el centro de la nube) y **proyecta** cada muestra sobre él (baja una perpendicular desde el punto hasta la recta).
2. La **dispersión (varianza)** de los puntos proyectados indica cuánta información conserva ese eje.
3. **Gira** el eje hasta que la varianza sea **máxima**: ese eje es la **PC1**. El perpendicular es la **PC2**.

El PCA es *solo un giro* de los ejes (no se pierde información); la **reducción de dimensionalidad** aparece cuando conservamos solo las primeras PC. Con miles de genes la receta es exactamente la misma.

**Qué hacer:** hay dos ejemplos fijos en monocitos: un par de genes **correlacionados** (`ISG15` y `MX1`, ambos activados por interferón) y un par **no correlacionados** (`IDO1` e `IL1B`, dos rutas distintas).
Gira el eje con el deslizador y observa los paneles ①②③. Marca *mostrar PC1 y PC2* o pulsa *ir a PC1*.

In [ ]:
#@title ▶ Ejercicio 8 · PCA, la idea { display-mode: "form" }
try:
    exercise_pca_geometry()
except NameError:
    print('⚠️ Ejecuta primero la celda de Preparación al principio del cuaderno (▶), espera a ✅ Listo y vuelve a ejecutar esta celda.')


#### ❓ Preguntas
1. **Par correlacionado**: ¿con qué ángulo conserva el eje más varianza? ¿Cuánta? ¿Cómo es el eje respecto a la nube de puntos?
2. **Par no correlacionado**: ¿cuánto cambia la varianza conservada con el ángulo? ¿Podría un eje resumir ambos genes?
3. En el panel ③, con el ángulo de la PC1, ¿cuál es la correlación entre las dos nuevas coordenadas? Si descartamos la PC2, ¿cuánta varianza perdemos en cada par?

<details>
<summary>💡 Respuestas</summary>

1. Con unos **36°** el eje conserva el **98 %** de la varianza total: corre a lo largo de la dirección larga de la nube alargada (los dos genes suben y bajan juntos, r = 0.95). Un número por muestra (la posición a lo largo de la PC1) puede sustituir a los dos genes.
2. Muy poco: entre el **45 % y el 55 %** sea cual sea el ángulo (el mejor eje, la PC1, conserva el 55 %). La nube no tiene dirección preferida, así que ningún eje resume ambos genes: perderíamos casi la mitad de la varianza.
3. **Alrededor de 0**: PC1 y PC2 están incorrelacionadas por construcción. Descartar la PC2 pierde solo el **2 %** de la varianza en el par correlacionado, pero el **45 %** en el no correlacionado. El PCA comprime bien *cuando los genes están correlacionados*: con miles de genes que responden juntos (todos los de interferón, todos los de monocito…) unas pocas PC resumen la mayor parte de la variación.

</details>

## Ejercicio 9 · PCA de todas las muestras: ¿qué determina la variación?
⏱ *8 min*

Cada una de las 4,224 muestras se describe con los **2,000 genes más variables**. El PCA da a cada muestra nuevas coordenadas (PC1, PC2…), ordenadas por la varianza que contienen.
El PCA es **no supervisado**: no sabe nada de donantes, tipos celulares o citocinas. Para saber qué *significa* una PC, **coloreamos las muestras por sus metadatos** y vemos qué coloración se alinea con los ejes.
El panel de **R²** lo cuantifica: la fracción de la varianza de cada PC que explica una variable (1 = por completo).

**Qué hacer:** el gráfico empieza coloreado por **donante**. Cambia la coloración (tipo celular, control vs tratado, citocina, número de células, tamaño de librería, un gen) y las PC de los ejes.

In [ ]:
#@title ▶ Ejercicio 9 · PCA de todas las muestras { display-mode: "form" }
try:
    exercise_pca_all()
except NameError:
    print('⚠️ Ejecuta primero la celda de Preparación al principio del cuaderno (▶), espera a ✅ Listo y vuelve a ejecutar esta celda.')


#### ❓ Preguntas
1. Con la coloración por defecto (donante), ¿se separan los donantes en PC1 y PC2? Colorea por **tipo celular**: ¿qué separan PC1 y PC2 y cuánta varianza contienen?
2. ¿Qué PC separa mejor a los donantes? (prueba distintas PC en los ejes: el panel de R² indica cuánto tiene cada una que ver con los donantes; luego colorea por donante.) ¿Y qué PC tiene mayor R² con las citocinas?
3. ¿Entonces las citocinas no tienen efecto sobre las células? ¿Qué significa el R² de la PC2 con el *tamaño de librería*: debe preocuparnos?

<details>
<summary>💡 Respuestas</summary>

1. No: los donantes están mezclados. Coloreado por tipo celular, la **PC1 (66 % de la varianza)** separa los **monocitos** de los linfocitos y la **PC2 (17 %)** separa las **células NK** de las T; T CD4 y T CD8 son el par más cercano. El R² del tipo celular con PC1 y PC2 es 0.99 y 0.98.
2. Los donantes se separan en la **PC6 (R² = 0.92)**, la PC3 (0.72) y la PC8 (0.65): prueba PC3 vs PC6. Las citocinas tienen un R² modesto en cualquier PC (como mucho 0.13, en PC3 y PC10): importan, pero en PC pequeñas.
3. **¡No!** El PCA muestra primero las diferencias **mayores**, y la diferencia entre un monocito y una célula T es mucho mayor que el efecto de cualquier citocina. Los efectos de las citocinas están en PC posteriores o **dentro de cada tipo celular** (siguiente ejercicio). El R² de la PC2 con el tamaño de librería (0.43) es el tipo celular disfrazado: las NK simplemente tienen librerías más pequeñas. Tras la normalización de la Parte C, la profundidad no gobierna las PC.

</details>

## Ejercicio 10 · PCA dentro de un tipo celular: respuestas a citocinas
⏱ *10 min*

Las diferencias entre tipos celulares ocultan casi todo lo demás. Un remedio habitual es analizar **un tipo celular cada vez**: seleccionamos sus muestras, elegimos sus propios genes más variables y repetimos el PCA.
Dentro de un tipo celular, lo que varía son los donantes y las citocinas.

**Qué hacer:** elige un **tipo celular** (por defecto monocitos, el más sensible) y las **PC**. Colorea por **citocina** y marca las que quieras resaltar (PBS siempre en negro), o colorea por donante o por un gen.
Los gráficos de barras de la derecha muestran los **genes con mayor peso** (loadings) en cada eje: te dicen qué *significa* el eje.

In [ ]:
#@title ▶ Ejercicio 10 · PCA dentro de un tipo celular { display-mode: "form" }
try:
    exercise_pca_celltype()
except NameError:
    print('⚠️ Ejecuta primero la celda de Preparación al principio del cuaderno (▶), espera a ✅ Listo y vuelve a ejecutar esta celda.')


#### ❓ Preguntas
1. En el gráfico por defecto (PC1 vs PC4 en monocitos), ¿qué citocinas se alejan de la nube principal y en qué dirección? Mira los genes detrás de cada eje.
2. Cambia los ejes a PC1 vs PC2 y colorea por **donante**. ¿Qué domina las primeras PC dentro de un tipo celular? Compara el R² de donante y citocina en la tabla.
3. Colorea por la expresión de `UTY` (un gen del cromosoma Y) y prueba varias PC: ¿qué PC separa donantes masculinos y femeninos en monocitos? (el artículo habla de 6 hombres y 6 mujeres)

<details>
<summary>💡 Respuestas</summary>

1. **IFN-β** se mueve a la derecha, a lo largo de la PC1, donde los genes principales incluyen genes inducidos por interferón (`ACOD1`, `CCL8`, `CCL7`, `IDO1`); **IL-4 e IL-13** suben a lo largo de la PC4, cuyos genes principales incluyen `ALOX15` y `WNT5B` (dianas de IL-4). **IL-4 e IL-13 caen en el mismo sitio**: tenlo presente para el Ejercicio 12.
2. **Donantes**: las diferencias basales entre personas son la mayor variación dentro de un tipo celular (R² de donante 0.72 en PC1 y 0.81 en PC2, frente a 0.18 y 0.13 de citocina). Las citocinas aparecen más claras en PC posteriores (R² 0.32 en PC3, 0.38 en PC4): los efectos individuales son menores que las diferencias entre personas. Por eso compararemos cada citocina con el PBS del **mismo donante**.
3. **PC5** (correlación 0.87 entre `UTY` y la PC5). El sexo del donante es una de las mayores fuentes de variación en monocitos, aunque nada tenga que ver con las citocinas. (En los otros tipos celulares aparece en otra PC: PC2 en T CD4, PC3 en T CD8 y NK.)

</details>

---
# Parte E · Expresión diferencial: ¿qué genes cambian?

El PCA mostró que las citocinas mueven las muestras en direcciones concretas. La **expresión diferencial (DE)** hace la pregunta **gen a gen**:

> *¿La expresión del gen g con la citocina es distinta de su expresión con PBS, más de lo que esperaríamos por azar?*

Los resultados que vas a explorar se **calcularon de antemano** con **DESeq2** (ahora no se ejecuta nada): para cada uno de los 4 tipos celulares y cada una de las 87 citocinas, la comparación *citocina vs PBS* en los 12 donantes.

> 🧬 **Solo los genes inmunes.** Para que el análisis sea ligero y los gráficos legibles, la DE se hizo con **~500 genes del sistema inmune** (506 expresados): la unión de los conjuntos GO *cytokine-mediated signaling pathway* y *T cell activation* y los conjuntos Hallmark de *respuesta a interferón α / γ*. Cada comparación analiza ~450 de ellos.
> **¿Qué perdemos?** (1) La lista se construyó *a partir de genes que responden a citocinas*, así que la **fracción de DEG es mucho mayor** que en un análisis de todo el transcriptoma; no compares porcentajes. (2) **Los genes fuera de la lista no se pueden encontrar**, ni siquiera los clásicos (`ACTB`, `GBP1`, `ALOX15`, `NKG7`, `CCL17`… no están): las herramientas avisan cuando un gen no se analizó.

<details>
<summary>🔬 ¿Cómo se calcularon los resultados? (opcional)</summary>

* DESeq2 trabaja con los **conteos crudos** y los normaliza él mismo (una versión robusta del CPM).
* El **diseño** `~ donor + cytokine` es **pareado**: cada donante se compara consigo mismo, así que las diferencias entre personas se cancelan.
* Para cada gen ajusta un modelo **binomial negativo** (los conteos no son normales) y un **test de Wald** pregunta si el efecto de la citocina es mayor que el ruido.
* Como se analizan ~450 genes a la vez, los p-valores se **ajustan por comparaciones múltiples** (Benjamini–Hochberg, controlando la tasa de falsos descubrimientos, FDR): el **p-valor ajustado** es el que hay que usar.
* Los genes con muy pocos conteos (menos de 10 conteos en 12 de las 24 muestras) no se analizan.

</details>

Dos números describen cada gen:

| Número | Significado |
|---|---|
| **log₂ fold change** | el **tamaño** del cambio: +1 = se duplica (×2), +2 = ×4, −1 = se reduce a la mitad |
| **p-valor ajustado** | la **evidencia**: cuán sorprendente sería el cambio si la citocina no hiciera nada (pequeño = convincente) |

Un gen se considera **diferencialmente expresado (DEG)** cuando supera **ambos** umbrales.


## Ejercicio 11 · Volcano plots: los genes que responden
⏱ *12 min*

Un **volcano plot** muestra todos los genes a la vez: *x* = log₂ fold change (tamaño del efecto), *y* = −log₁₀ p-valor ajustado (evidencia; más alto = más convincente).
**Rojo** = DEG sobre-expresados, **azul** = DEG infra-expresados, **gris** = no significativos. Los genes arriba y lejos del centro son los más robustos y fuertes.

**Qué hacer:** por defecto, **IFN-γ en monocitos**, con los genes `CXCL9`, `CXCL10`, `IDO1`, `STAT1` e `IRF1` resaltados. Cambia el **tipo celular** y la **citocina**, los **umbrales** y los genes a resaltar
(escribe un símbolo o elige una lista predefinida; solo se pueden encontrar los ~500 genes inmunes); las tablas muestran su fold change, p-valor ajustado y expresión media.

In [ ]:
#@title ▶ Ejercicio 11 · Volcano plots { display-mode: "form" }
try:
    exercise_volcano()
except NameError:
    print('⚠️ Ejecuta primero la celda de Preparación al principio del cuaderno (▶), espera a ✅ Listo y vuelve a ejecutar esta celda.')


#### ❓ Preguntas
1. ¿Cuántos genes suben y bajan con IFN-γ en monocitos? Mira los cinco genes resaltados: ¿cuáles son DEG y cuántas veces más altos que en PBS? ¿Alguno es significativo pero *no* es DEG?
2. Pon el umbral de fold change a 0, 1 y 2 (p-valor ajustado 0.05), y después cambia el umbral del p-valor ajustado de 0.2 a 0.001 (fold change 0.5). ¿Qué umbral importa más? ¿Qué aporta el umbral de fold change?
3. Cambia la citocina a **IL-4** y resalta `CCL26`, `CD1C`, `STAT1`, `IDO1`. ¿Qué pasa con los genes de interferón?
4. Elige **células NK** e **IL-2**. ¿Qué tienen en común los genes más sobre-expresados `IL2RA`, `CISH` y `SOCS2`?

<details>
<summary>💡 Respuestas</summary>

1. **96 suben y 95 bajan** (de 452 genes analizados). `CXCL9` (×10.5), `IDO1` (×5.5) y `CXCL10` (×3.7) suben mucho: la clásica **respuesta a IFN-γ** (reclutamiento de células T y catabolismo del triptófano). `STAT1` **no** es DEG (×1.1, p ajustado = 0.34), aunque es el factor de transcripción de esa respuesta: ya es abundante en monocitos. `IRF1` es **significativo** (p ajustado ≈ 10⁻⁵) pero su cambio es pequeño (×1.4, log₂FC 0.49 < 0.5): significativo no es lo mismo que grande.
2. Umbral de fold change 0 / 1 / 2 → **311 / 70 / 14** DEG; p ajustado 0.2 / 0.001 → 197 / 161 DEG (con fold change 0.5; 191 con 0.05). **El umbral de fold change importa mucho más**: elimina cambios estadísticamente significativos pero biológicamente pequeños, y deja los que son fiables y grandes.
3. IL-4 sobre-expresa con fuerza sus dianas (`CCL26` ×207, `CD1C` ×21) y **reduce** los genes de interferón (`IDO1` ×0.1, `STAT1` ×0.4). IL-4 e IFN-γ empujan a los monocitos en sentidos opuestos.
4. Son genes de la **señalización de IL-2**: `IL2RA` es la cadena α del receptor de alta afinidad de IL-2, y `CISH` y `SOCS2` son dianas de STAT5 que retroalimentan para apagar la señal. Las células NK responden a IL-2 a través de STAT5.

</details>


## Ejercicio 12 · Comparar dos respuestas: ¿parecidas o distintas?
⏱ *10 min*

¿Activan dos citocinas los mismos genes? Para cada gen tenemos un log₂ fold change en la **comparación A** y en la **comparación B** (cada una = una citocina vs PBS en un tipo celular). En un diagrama de dispersión de A frente a B,
dos respuestas parecidas caen a lo largo de la **diagonal**, y el **coeficiente de correlación** lo resume: **r = 1** misma respuesta, **r = 0** sin relación, **r = −1** opuestas.
Los colores indican dónde es significativo el gen: **azul** solo en A, **naranja** solo en B, **violeta** en ambas, gris en ninguna. El panel derecho ordena las demás citocinas según lo que se parecen a A (en el mismo tipo celular).

**Qué hacer:** por defecto se comparan **IL-4 e IL-13 en monocitos**. Cambia la citocina y el tipo celular de A y B (¡pueden ser de tipos celulares distintos!), carga uno de los **ejemplos**, elige qué genes se usan para la correlación y **resalta genes**.

> ⚠️ Ambas comparaciones usan el **mismo control PBS**: el ruido de las muestras PBS acaba en las dos y puede crear o inflar una correlación. Cuidado con valores pequeños de r (≈ 0.2 o menos).

In [ ]:
#@title ▶ Ejercicio 12 · Comparar dos respuestas { display-mode: "form" }
try:
    exercise_compare()
except NameError:
    print('⚠️ Ejecuta primero la celda de Preparación al principio del cuaderno (▶), espera a ✅ Listo y vuelve a ejecutar esta celda.')


#### ❓ Preguntas
1. ¿Cuánto se parecen las respuestas a IL-4 e IL-13 en monocitos? Ahora cambia ambas comparaciones a **células T CD4**. ¿Qué pasa y por qué? (recuerda `IL13RA1` del Ejercicio 7)
2. Carga los ejemplos **IFN-β vs IFN-γ en monocitos** e **IFN-γ vs IL-4 en monocitos**. Compara las correlaciones y usa el panel de la derecha: ¿qué citocinas responden más como IFN-γ?
3. Carga **TNF-α en monocitos vs en células T CD4** e **IL-6 en monocitos vs en células T CD4**: ¿qué estímulo desencadena una respuesta más parecida en los dos tipos celulares? ¿Y **TNF-α vs IL-6** en monocitos?
4. Cambia *genes usados para la correlación* a "genes significativos en A o en B". ¿Cuándo cambia mucho r y cuándo no?

<details>
<summary>💡 Respuestas</summary>

1. En monocitos las dos respuestas son **casi idénticas (r = 0.97)** e IL-13 es la citocina más parecida a IL-4 de 86. En células T CD4 **r cae a 0.41** (IL-13 es solo la 3.ª). IL-4 e IL-13 comparten receptor (IL-4Rα/IL-13Rα1), pero las células T apenas expresan `IL13RA1` (~2 CPM frente a ~256 en monocitos), así que responden a IL-4 y no a IL-13. **El mismo par de estímulos puede parecer idéntico o muy distinto según el tipo celular**.
2. **IFN-β vs IFN-γ: r = 0.78** (ambos interferones activan STAT1 y los genes estimulados por interferón); **IFN-γ vs IL-4: r = −0.35**: programas opuestos (lo que uno sube, el otro lo baja). El ranking de IFN-γ en monocitos empieza con IL-12 (0.87), IFN-β e IFN-ω (0.78) e IL-15 (0.74) y termina con IL-17E y TNF-α (−0.46, −0.45).
3. **TNF-α** da una respuesta parecida en los dos tipos celulares (**r = 0.70**), **IL-6** una más específica de tipo celular (**r = 0.47**). **TNF-α vs IL-6 en monocitos: r = 0.22**, una relación débil. Mira siempre los datos antes de suponer que dos citocinas comparten un programa.
4. En monocitos, donde ambas citocinas cambian muchos genes, r apenas se mueve (IL-4 vs IL-13: 0.97 → 0.97). Donde las respuestas son más débiles, restringirse a genes que responden (células T CD4: 0.41 → 0.51; TNF-α vs IL-6 en monocitos: 0.22 → 0.28) elimina ruido y r **aumenta**. Pero también *selecciona* genes que cambian, así que la comparación deja de estar libre de sesgo: indica qué genes se usaron. Recuerda además que todo el análisis usa ya una lista de genes inmunes, enriquecida en genes que responden a citocinas: las correlaciones son mayores que en todo el genoma.

</details>


---
# 🏁 Cierre

Has pasado de una sola variable a comparar programas transcripcionales:

| Ejercicio | La idea a recordar |
|:--:|---|
| 1–2 | Una variable es un nombre para un valor; cada tipo de gráfico responde a una pregunta. |
| 3 | AnnData mantiene **alineados** los números (`X`), las muestras (`obs`) y los genes (`var`); `adata[muestras, genes]` selecciona un subconjunto. |
| 4 | Los conteos crudos mezclan biología con la **cantidad de material** (tamaño de librería). |
| 5 | El **CPM** divide por el tamaño de librería para que las muestras sean comparables. |
| 6 | El **logaritmo** comprime la escala, convierte cocientes en diferencias y evita que unos pocos genes dominen. |
| 7 | Los **mapas de calor** muestran patrones; solo tienen sentido *tras* normalizar. |
| 8–10 | El **PCA** es un giro que encuentra las direcciones de mayor varianza. Colorear por metadatos dice qué significa cada PC: tipo celular ≫ donante ≫ citocina. Mira **dentro de un tipo celular** para ver los efectos de las citocinas. |
| 11 | Un **DEG** necesita evidencia (**p-valor ajustado**) *y* un efecto relevante (**fold change**). El diseño pareado compara cada donante consigo mismo. |
| 12 | La **correlación de fold changes** dice si dos estímulos activan programas parecidos, y **depende del tipo celular**. |

### Para seguir
* **Enriquecimiento funcional** de las listas de DEG (¿qué rutas?): `gseapy`, `decoupler`.
* **Análisis de célula única** de los datos originales: `scanpy`; buenas prácticas en Heumos *et al.* (2023).
* El propio **Human Cytokine Dictionary** y su paquete `huCIRA` (Oesinghaus *et al.*, 2025) para interpretar tus datos con firmas de respuesta a citocinas.

### 📝 Cuestionario rápido
Tu profesor mostrará el **código QR** del cuestionario (3 preguntas, 1 minuto).

### Referencias
* Oesinghaus L. *et al.* (2025) A single-cell cytokine dictionary of human peripheral blood. *bioRxiv*, doi:10.64898/2025.12.12.693897
* Virshup I. *et al.* (2024) anndata: Access and store annotated data matrices. *J. Open Source Softw.* 9:4371
* Heumos L. *et al.* (2023) Best practices for single-cell analysis across modalities. *Nat. Rev. Genet.* 24:550–572
* Love M.I., Huber W., Anders S. (2014) Moderated estimation of fold change and dispersion for RNA-seq data with DESeq2. *Genome Biol.* 15:550
* Muzellec B. *et al.* (2023) PyDESeq2: a python package for bulk RNA-seq differential expression analysis. *Bioinformatics* 39:btad547
* Benjamini Y., Hochberg Y. (1995) Controlling the false discovery rate. *J. R. Stat. Soc. B* 57:289–300
